https://tonikph-my.sharepoint.com/:p:/g/personal/bbanik_tonikbank_com/IQAvsct_VJYKRoYRQISh8jy1AWHNdHcuDqk9bIfwQjpiW6M?e=wvYx82

# Import packages

In [1]:

# %% [markdown]
# # Jupyter Notebook Loading Header
#
# This is a custom loading header for Jupyter Notebooks in Visual Studio Code.
# It includes common imports and settings to get you started quickly.
# %% [markdown]
## Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.cloud import bigquery
from google.cloud import storage
import os
import tempfile
import time
from datetime import datetime
import uuid
import joblib
import uuid
from sklearn.metrics import roc_auc_score
from datetime import datetime, timedelta
import gcsfs
import duckdb as dd
import pickle
import joblib
from typing import Union
import io
path = r'C:\Users\Dwaipayan\AppData\Roaming\gcloud\legacy_credentials\dchakroborti@tonikbank.com\adc.json'
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = path
client = bigquery.Client(project='prj-prod-dataplatform')
os.environ["GOOGLE_CLOUD_PROJECT"] = "prj-prod-dataplatform"
# %% [markdown]
## Configure Settings
# Set options or configurations as needed
pd.set_option('display.max_columns', None)
pd.set_option("Display.max_rows", 100)
sns.set_style('whitegrid')

In [2]:
import pandas as pd
import numpy as np
import os
import pickle

from google.cloud import bigquery
from google.cloud import storage

from sklearn.metrics import roc_auc_score

# Settings

In [3]:
pd.set_option('display.max_columns', 100)

In [4]:
GS_BUCKET = "prod-asia-southeast1-tonik-aiml-workspace"
PROJECT_ID = "prj-prod-dataplatform"
RANDOM_SEED = 2024

# Functions

In [5]:
def define_cat_features(columns, cat_vars):
    return list(set(cat_vars).intersection(columns))

def generate_bucket_url(filename, bucket_name):

    return f"gs://{bucket_name}/{filename}"


def save_to_gcs(data, filename, bucket_name):
    """
    Save data to Google Cloud Storage bucket.

    :param data: The data to save. Can be a string, bytes, or a file-like object.
    :param filename: The name of the file to save in the bucket.
    :param bucket_name: The name of the GCS bucket. Defaults to 'ABC'.
    :return: The public URL of the saved file.
    """
    # Create a client
    client = storage.Client()

    # Get the bucket
    bucket = client.bucket(bucket_name)

    # Create a blob (file) in the bucket
    blob = bucket.blob(filename)

    # Determine the content type and upload accordingly
    if isinstance(data, str):
        blob.upload_from_string(data)
    elif isinstance(data, bytes):
        blob.upload_from_string(data, content_type='application/octet-stream')
    elif hasattr(data, 'read'):  # File-like object
        blob.upload_from_file(data)
    else:
        raise ValueError("Unsupported data type. Please provide a string, bytes, or file-like object.")


    print(f"File {filename} uploaded to {bucket_name}.")
    return blob.public_url


def load_artifact_from_gcs(filename, bucket_name):
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(filename)

    # Download model
    artifact_bytes = blob.download_as_bytes()
    artifact = pickle.loads(artifact_bytes)
    print(f"Model loaded from gs://{bucket_name}/{filename}")

    return artifact


def load_from_gcs(filename, bucket_name, output_type='bytes'):
    """
    Load data from Google Cloud Storage bucket with flexible output formats.

    :param filename: The name of the file to load from the bucket.
    :param bucket_name: The name of the GCS bucket.
    :param output_type: The desired output format. Can be 'bytes', 'string', 'pickle', or 'file'.
                       'bytes': Returns raw bytes
                       'string': Returns decoded string
                       'pickle': Returns unpickled Python object
                       'file': Returns a file-like object for streaming
    :return: The loaded data in the specified format.
    """
    import pickle
    import io

    # Create a client
    client = storage.Client()

    # Get the bucket and blob
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(filename)

    # Handle different output types
    if output_type == 'bytes':
        return blob.download_as_bytes()

    elif output_type == 'string':
        return blob.download_as_string().decode('utf-8')

    elif output_type == 'pickle':
        pickled_data = blob.download_as_bytes()
        return pickle.loads(pickled_data)

    elif output_type == 'file':
        file_obj = io.BytesIO()
        blob.download_to_file(file_obj)
        file_obj.seek(0)  # Reset file pointer to beginning
        return file_obj

    else:
        raise ValueError("Unsupported output_type. Must be one of: 'bytes', 'string', 'pickle', 'file'")

def load_artifacts_logreg(exp_number):

    model_filename = f'Oleh/tendo/experiments/{exp_number}/model.pkl'
    model = load_artifact_from_gcs(model_filename, GS_BUCKET)

    feature_filename = f'Oleh/tendo/experiments/{exp_number}/features.pkl'
    features = load_artifact_from_gcs(feature_filename, GS_BUCKET)

    scaler_filename = f'Oleh/tendo/experiments/{exp_number}/scaler.pkl'
    scaler = load_artifact_from_gcs(scaler_filename, GS_BUCKET)

    return model, features, scaler

def save_artifact_to_gcs(artifact, filename, bucket_name):
    """Saves the Cox model to Google Cloud Storage."""
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(filename)

    # Serialize artifact
    artifact_bytes = pickle.dumps(artifact)

    # Upload to GCS
    blob.upload_from_string(artifact_bytes)
    print(f"Artifact saved to gs://{bucket_name}/{filename}")

In [6]:
import pandas as pd
from google.cloud import bigquery
from sklearn.metrics import roc_auc_score
from typing import Dict

def calculate_gini_for_table(
    data: pd.DataFrame,
    date_column: str,
    score_column: str,
    target_column: str,
    data_periods_dict: Dict,
    weights_col: str = None
):
    """
    Calculate Gini coefficient for different time periods.

    Args:
        project_id: BigQuery project ID
        table_name: Full table name (dataset.table)
        date_column: Name of the date column
        score_column: Name of the score column
        target_column: Name of the target column
        target_maturity_column: Name of the target maturity column
        data_periods_dict: Dictionary with periods, e.g.:
            {'Train': {'start': '2024-01-01', 'end': '2025-01-31'},
             'Test': {'start': '2025-02-01', 'end': '2025-12-31'}}

    Returns:
        pandas.DataFrame: Table with Gini coefficients for each period
    """
    dt = data[data[target_column].notna()].copy()

    # Convert date column to datetime and extract just the date part
    dt[date_column] = pd.to_datetime(dt[date_column]).dt.date

    # Initialize results
    gini_results = []

    print("Gini Coefficient Results:")
    print("=" * 50)

    # Calculate Gini for each period
    for period_name, period_info in data_periods_dict.items():
        start_date = pd.to_datetime(period_info['start']).date()
        end_date = pd.to_datetime(period_info['end']).date()

        # Filter data for the current period
        period_mask = (dt[date_column] >= start_date) & (dt[date_column] <= end_date)
        period_data = dt[period_mask].copy()

        sample_size = period_data['ee_customer_id'].nunique()
        bad_rate = 100*(1 - period_data[['ee_customer_id',target_column]].drop_duplicates()[target_column].sum() / sample_size)

        if len(period_data) == 0:
            print(f"{period_name}: No data available for period {start_date} to {end_date}")
            gini_results.append({
                'Period': period_name,
                'Start_Date': start_date,
                'End_Date': end_date,
                'Sample_Size': 0,
                'Bad Rate': np.nan,
                'Gini_Coefficient': None
            })
            continue

        # Check if we have both classes (0 and 1) in target
        unique_targets = period_data[target_column].unique()
        if len(unique_targets) < 2:
            print(f"{period_name}: Only one class present in target variable. Cannot calculate Gini.")
            gini_results.append({
                'Period': period_name,
                'Start_Date': start_date,
                'End_Date': end_date,
                'Sample_Size': sample_size,
                'Bad Rate': bad_rate,
                'Gini_Coefficient': None
            })
            continue

        # Calculate Gini coefficient
        try:
            if weights_col:
                auc = roc_auc_score(period_data[target_column], period_data[score_column], sample_weight=period_data[weights_col])
                gini = 2 * auc - 1
            else:
                auc = roc_auc_score(period_data[target_column], period_data[score_column])
                gini = 2 * auc - 1

            print(f"{period_name}: {round(gini, 4)} (Sample size: {len(period_data):,})")

            gini_results.append({
                'Period': period_name,
                'Start_Date': start_date,
                'End_Date': end_date,
                'Sample_Size': sample_size,
                'Bad Rate': bad_rate,
                'Gini_Coefficient': round(gini, 4)
            })

        except Exception as e:
            print(f"{period_name}: Error calculating Gini - {str(e)}")
            gini_results.append({
                'Period': period_name,
                'Start_Date': start_date,
                'End_Date': end_date,
                'Sample_Size': sample_size,
                'Bad Rate': bad_rate,
                'Gini_Coefficient': None
            })

    # Create results DataFrame
    results_df = pd.DataFrame(gini_results)

    print("\n" + "=" * 50)
    print("Summary Table:")
    print(results_df.to_string(index=False))

    return results_df

# Data loading

In [7]:
client = bigquery.Client(PROJECT_ID)

In [8]:
# PROD API
sql_query_prod_api = """
SELECT
  employee_id as ee_customer_id,
  run_date,
  ee_attrition_risk_segment as attrition_risk_segment,
  ee_attrition_time_to_leave as attrition_time_to_leave,
  oop_score as oop_score_prod,
  oop_risk_segment as oop_risk_segment_prod
FROM `prj-prod-dataplatform.tendo_mart.tendo_scorecard_master_table_api`
"""

dt_prod_api = client.query(sql_query_prod_api).to_dataframe()

In [9]:
dt_prod_api.head(2)

,ee_customer_id,run_date,attrition_risk_segment,attrition_time_to_leave,oop_score_prod,oop_risk_segment_prod
0,1532098,2026-01-13T08:19:00.740172,10-12,10,0.368567,E
1,1532098,2026-01-13T08:43:49.924701,10-12,10,0.368567,E


In [10]:
# PROD BATCH
sql_query_prod_batch = """
SELECT
  employee_id as ee_customer_id,
  run_date,
  ee_attrition_risk_segment as attrition_risk_segment,
  ee_attrition_time_to_leave as attrition_time_to_leave,
  oop_score as oop_score_prod,
  oop_risk_segment as oop_risk_segment_prod
FROM `prj-prod-dataplatform.tendo_mart.tendo_scorecard_master_table`
"""

dt_prod_batch = client.query(sql_query_prod_batch).to_dataframe()

In [11]:
dt_prod_batch.head(2)

,ee_customer_id,run_date,attrition_risk_segment,attrition_time_to_leave,oop_score_prod,oop_risk_segment_prod
0,1157338,2025-10-14,Low,10,0.704469,A
1,1456942,2025-10-14,Low,10,0.735589,A


In [12]:
# OOP Latest targets
sql_query_oop_targets = """
SELECT
  user_id as ee_customer_id,
  target as oop_target
FROM `prj-prod-dataplatform.tendo_mart.tendo_collection_target_master`
WHERE target_maturity_flag = 1
"""

dt_oop_targets = client.query(sql_query_oop_targets).to_dataframe()

In [13]:
dt_oop_targets.head(2)

,ee_customer_id,oop_target
0,747943,0
1,752636,0


In [14]:
dt_raw = pd.read_pickle(
    generate_bucket_url('Oleh/tendo/data/raw_data_14012026.pkl', GS_BUCKET)
)

In [15]:
dt_raw.head(2)

,ee_customer_id,ee_email,ee_phone_number,ee_firstname,ee_middlename,ee_lastname,ee_birthdate,ee_gender,ee_email_verified_flag,ee_telephone_verified_flag,ee_id_verified_flag,ee_income_verified_flag,ee_morning_time_contact_time,ee_afternoon_time_contact_time,ee_account_activated_flag,ee_region_name,ee_city_name,ee_barangay,ee_address_line_1,ee_address_line_2,ee_postal_code,ee_landmark,ee_residing_date,ee_employment_status,ee_civil_status,ee_kyc_doc_name,ee_is_citizen_flag,ee_onboarding_date,ee_employment_date,ee_fraud_status,ee_job_type,ee_comment,ee_department,ee_recommended_ir,ee_job_title,ee_employment_type,ee_nature_of_work,ee_permanent_freeze_date,ee_resignation_date,ee_product_type,ee_frozen_status,cust_risk_employer_time_score_v1,cust_risk_gender_score_v1,cust_risk_declared_income_score_v1,cust_risk_civil_status_score_v1,cust_risk_combined_score_v1,cust_risk_cat_v1,er_employer_id,er_employer_group_id,er_employer_name,er_email_domain,er_repayment_days_month,er_custom_email,er_payment_reminders,er_address,er_postal_code_id,er_max_base_interest_rate,er_employer_industry,er_employer_status,er_activated_at,er_deleted_at,er_created_at,er_updated_at,cl_monthly_utility_bills_amount,cl_monthly_income_gross,cl_monthly_income_net,cl_multiple_purchases_enabled_flag,cl_max_credit_limit_multiplier,cl_max_debt_income_ratio,cl_matured_fpd30_flag,cl_matured_fspd30_flag,cl_matured_fstpd30_flag,cl_matured_fstfpd30_flag,cl_fpd30_flag,cl_fspd30_flag,cl_fstpd30_flag,cl_fstfpd30_flag,ln_loan_id,ln_loan_type,ln_loan_status,ln_disbursement_channel,ln_original_principal,ln_orig_interest_fees,ln_orig_tenor,ln_loan_application_datetime,ln_repaid_full_flag,ln_fully_repaid_date,ln_fpd30_flag,ln_fspd30_flag,ln_fstpd30_flag,ln_fstfpd30_flag,ln_min_loan_due_date,ln_os_principal_at_fpd30,ln_os_principal_at_fspd30,ln_os_principal_at_fstpd30,ln_os_principal_at_fstfpd30,ln_matured_fpd30_flag,ln_matured_fspd30_flag,ln_matured_fstpd30_flag,ln_matured_fstfpd30_flag
0,1165907,efrengoyone97@gmail.com,+63 (906) 138 7568,Efren,Orgong,Goyone,1972-06-08,Male,0,1,2,2,None,None,2,None,None,None,Guava St Aratilles Masville Sucat Barangay BF ...,None,None,Near Aratilles Covered Court,None,regular,Married,None,true,2024-03-04 11:34:33+00:00,2021-06-01,Risk,None,None,None,None,Assembler,Permanent Job (Private sector),None,NaT,None,employer,Frozen,139,118,110,127,494,B,10216.000000000,None,Hino Motors Philippines Corporation,None,"[""15"",""30""]",1,-1,None,None,3.5,40,PARKED,2024-02-16 20:10:14.000000,NaT,2024-02-16 20:10:14+00:00,2025-12-09 10:50:23+00:00,700,18500,16000,1,188,30,1,1,1,1,0,1,1,1,1508451,Tendo,Approved/Disbursed,PH_MET,10000.0,3600.0,12,2024-05-19 16:53:18+00:00,1,2025-06-15,0,1,1,1,2024-06-30,0.0,9382.999,0.0,0.0,1,1,1,1
1,739181,marellpadios30@gmail.com,+63 (956) 760 3153,Ma Ellalita,Padios,Balingasa,1985-09-30,Female,1,1,2,2,None,None,2,Region IV-A,None,BAGONG NAYON,Road B lot 20 phase Cogeo Village,None,1870,None,None,regular,Married,None,true,2022-08-09 13:32:10+00:00,2021-10-21,Risk,None,"110,Resigned",None,None,customer service hero,Permanent Job (Private sector),None,2023-09-14 18:48:34+00:00,2023-07-30,employer,Frozen,98,120,110,127,455,E,10049.000000000,None,Intelegencia,None,"[""15"",""30""]",1,8,"OTC Building, 23 Sumulong Hwy",475,4.0,8,IN_PROGRESS,None,NaT,2020-07-30 12:29:36+00:00,2025-12-11 17:19:20+00:00,2500,20000,16000,0,80,25,1,1,1,1,1,1,1,1,609494,Tendo,Approved/Disbursed,None,1000.0,120.0,3,2023-02-09 07:16:24+00:00,1,2023-05-30,0,0,0,0,2023-03-15,0.0,0.000,0.0,0.0,1,1,1,1


In [16]:
dt_raw['ee_customer_id'] = dt_raw['ee_customer_id'].astype('str')
dt_raw['ee_onboarding_date'] = pd.to_datetime(dt_raw['ee_onboarding_date']).dt.tz_localize(None)

# OOP

In [17]:
# BS OOP new
sql_query_oop_new = """
SELECT *
FROM `prj-prod-dataplatform.risk_mart.tendo_backscored_new_users_jan23_jan26_20260201_oop_with_osbal`
"""

dt_bs_oop_new = client.query(sql_query_oop_new).to_dataframe()

# BS OOP existing
sql_query_oop_existing = """
SELECT *
FROM `prj-prod-dataplatform.risk_mart.tendo_backscored_existing_users_jan23_jan26_20260201_oop`
"""

dt_bs_oop_ex = client.query(sql_query_oop_existing).to_dataframe()

`Don't have the osbal table calculation. Need to ask Oleh or understand the process of understanding how to calculate the outstanding balance data`

Biswa  Oleh, for now just use the scorecard v2.2 as of January Employment date fix for backscoring

I understand, but it is a lot of work, starting from requesting to update dev tables, generating data and backscore it. 
 
I am also thinking that it would be a good idea to score every customer in prod, but to use outputs only for customers eligible for sc2.0. So in that case we will not need to have backscored scores at all. 
 
Biswa
Oleh, for now just use the scorecard v2.2 as of January Employment date fix for backscoring

I can generate those backscored tables by the EOW
 
Hi Udhayanan Agasthiappan
Please update
 
prj-prod-dataplatform.worktable_data_analysis.tendo_scorecard_features_data_23-02-2026

prj-prod-dataplatform.worktable_data_analysis.tendo_scorecard_loan_repayment_data_23-02-2026

In [18]:
dt_bs_oop_new['ee_customer_id'] = dt_bs_oop_new['ee_customer_id'].astype('str')
dt_bs_oop_ex['ee_customer_id'] = dt_bs_oop_ex['ee_customer_id'].astype('str')

In [19]:
dt_bs_oop_new.head(2)

,ee_customer_id,calc_date,target_dev,score_oop,osbal_as_of_resignation_date,osbal_as_of_oop_eligible_date,osbal_as_of_current_date
0,1211049,2024-07-01,NaN,0.104421,NaN,NaN,0.003
1,1194175,2024-06-01,NaN,0.105540,NaN,NaN,7935.203


In [20]:
dt_bs_oop_ex.head(2)

,ee_customer_id,calc_date,target_dev,score_oop
0,102999,2023-01-01,NaN,0.546834
1,103347,2023-01-01,NaN,0.657887


In [21]:
dt_bs_oop_ex["calc_date"] = pd.to_datetime(dt_bs_oop_ex["calc_date"], errors="coerce")
dt_bs_oop_ex["calc_date_correct"] = pd.to_datetime(dt_bs_oop_ex["calc_date"], errors="coerce") - pd.DateOffset(months=1)
dt_bs_oop_ex['calc_date_ym'] = dt_bs_oop_ex['calc_date_correct'].dt.year*100 + dt_bs_oop_ex['calc_date_correct'].dt.month

In [22]:
dt_bs_oop_ex.head(2)

,ee_customer_id,calc_date,target_dev,score_oop,calc_date_correct,calc_date_ym
0,102999,2023-01-01,NaN,0.546834,2022-12-01,202212
1,103347,2023-01-01,NaN,0.657887,2022-12-01,202212


## Gini

### prod new

In [23]:
dt_prod_api = dt_prod_api.merge(
    dt_raw[['ee_customer_id','ee_onboarding_date']].drop_duplicates(),
    how='left',
    on='ee_customer_id'
).merge(
    dt_oop_targets,
    how='left',
    on='ee_customer_id'
).merge(
    dt_bs_oop_new[['ee_customer_id', 'score_oop', 'osbal_as_of_resignation_date', 'osbal_as_of_oop_eligible_date', 'osbal_as_of_current_date']],
    how='left',
    on='ee_customer_id'
)

In [24]:
dt_prod_api['onb_rd_diff'] = (abs(pd.to_datetime(dt_prod_api['run_date']).dt.normalize() - pd.to_datetime(dt_prod_api['ee_onboarding_date']).dt.normalize())).dt.days
dt_prod_api['ee_onboarding_date'] = pd.to_datetime(dt_prod_api['ee_onboarding_date']).dt.normalize()
dt_prod_api['run_date'] = pd.to_datetime(dt_prod_api['run_date']).dt.normalize()
dt_prod_api['onboarding_date_ym'] = dt_prod_api['ee_onboarding_date'].dt.year*100 + dt_prod_api['ee_onboarding_date'].dt.month
dt_prod_api['run_date_ym'] = dt_prod_api['run_date'].dt.year*100 + dt_prod_api['run_date'].dt.month

In [25]:
dt_prod_api_calc = (dt_prod_api
                    .dropna(subset=['ee_onboarding_date','oop_target'])
                    .sort_values(['onb_rd_diff','run_date'])
                    .drop_duplicates(subset=['ee_customer_id'], keep='first'))

In [26]:
dt_prod_api_calc.head(1)

,ee_customer_id,run_date,attrition_risk_segment,attrition_time_to_leave,oop_score_prod,oop_risk_segment_prod,ee_onboarding_date,oop_target,score_oop,osbal_as_of_resignation_date,osbal_as_of_oop_eligible_date,osbal_as_of_current_date,onb_rd_diff,onboarding_date_ym,run_date_ym
48438,1462050,2025-06-20,10-12,10,0.548142,B,2025-06-20,0,0.526408,17378.052,17378.052,17378.052,0.0,202506.0,202506


In [27]:
print('OOP New Users Prod')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Jun 2025 - Sep 2025': {'start': '2025-06-01', 'end': '2025-09-30'}
}

calculate_gini_for_table(
    data = dt_prod_api_calc,
    date_column = 'ee_onboarding_date',
    score_column = 'oop_score_prod',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict
)

OOP New Users Prod

Gini Coefficient Results:
Jun 2025: 0.25 (Sample size: 15)
Jul 2025: -0.0292 (Sample size: 110)
Aug 2025: -0.0837 (Sample size: 63)
Sep 2025: -0.2605 (Sample size: 92)
Jun 2025 - Sep 2025: -0.1034 (Sample size: 280)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30           15 53.333333            0.2500
           Jul 2025 2025-07-01 2025-07-31          110 65.454545           -0.0292
           Aug 2025 2025-08-01 2025-08-31           63 63.492063           -0.0837
           Sep 2025 2025-09-01 2025-09-30           92 68.478261           -0.2605
Jun 2025 - Sep 2025 2025-06-01 2025-09-30          280 65.357143           -0.1034


,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,15,53.333333,0.2500
1,Jul 2025,2025-07-01,2025-07-31,110,65.454545,-0.0292
2,Aug 2025,2025-08-01,2025-08-31,63,63.492063,-0.0837
3,Sep 2025,2025-09-01,2025-09-30,92,68.478261,-0.2605
4,Jun 2025 - Sep 2025,2025-06-01,2025-09-30,280,65.357143,-0.1034


In [28]:
print('OOP New Users Prod BS')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Jun 2025 - Sep 2025': {'start': '2025-06-01', 'end': '2025-09-30'}
}

calculate_gini_for_table(
    data = dt_prod_api_calc[dt_prod_api_calc['score_oop'].notna()],
    date_column = 'ee_onboarding_date',
    score_column = 'score_oop',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict
)

OOP New Users Prod BS

Gini Coefficient Results:
Jun 2025: 0.0357 (Sample size: 15)
Jul 2025: 0.262 (Sample size: 103)
Aug 2025: -0.0646 (Sample size: 60)
Sep 2025: -0.208 (Sample size: 90)
Jun 2025 - Sep 2025: 0.0012 (Sample size: 268)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30           15 53.333333            0.0357
           Jul 2025 2025-07-01 2025-07-31          103 65.048544            0.2620
           Aug 2025 2025-08-01 2025-08-31           60 63.333333           -0.0646
           Sep 2025 2025-09-01 2025-09-30           90 67.777778           -0.2080
Jun 2025 - Sep 2025 2025-06-01 2025-09-30          268 64.925373            0.0012


,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,15,53.333333,0.0357
1,Jul 2025,2025-07-01,2025-07-31,103,65.048544,0.2620
2,Aug 2025,2025-08-01,2025-08-31,60,63.333333,-0.0646
3,Sep 2025,2025-09-01,2025-09-30,90,67.777778,-0.2080
4,Jun 2025 - Sep 2025,2025-06-01,2025-09-30,268,64.925373,0.0012


In [29]:
print('OOP New Users Prod, weighted as-is')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Jun 2025 - Sep 2025': {'start': '2025-06-01', 'end': '2025-09-30'}
}

calculate_gini_for_table(
    data = dt_prod_api_calc[dt_prod_api_calc['osbal_as_of_oop_eligible_date'].notna()],
    date_column = 'ee_onboarding_date',
    score_column = 'oop_score_prod',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col='osbal_as_of_oop_eligible_date'
)

OOP New Users Prod, weighted as-is

Gini Coefficient Results:
Jun 2025: 0.633 (Sample size: 13)
Jul 2025: 0.0109 (Sample size: 84)
Aug 2025: -0.1007 (Sample size: 43)
Sep 2025: -0.6702 (Sample size: 63)
Jun 2025 - Sep 2025: -0.1629 (Sample size: 203)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30           13 61.538462            0.6330
           Jul 2025 2025-07-01 2025-07-31           84 71.428571            0.0109
           Aug 2025 2025-08-01 2025-08-31           43 76.744186           -0.1007
           Sep 2025 2025-09-01 2025-09-30           63 76.190476           -0.6702
Jun 2025 - Sep 2025 2025-06-01 2025-09-30          203 73.399015           -0.1629


,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,13,61.538462,0.6330
1,Jul 2025,2025-07-01,2025-07-31,84,71.428571,0.0109
2,Aug 2025,2025-08-01,2025-08-31,43,76.744186,-0.1007
3,Sep 2025,2025-09-01,2025-09-30,63,76.190476,-0.6702
4,Jun 2025 - Sep 2025,2025-06-01,2025-09-30,203,73.399015,-0.1629


In [30]:
dt_prod_api_calc['osbal_as_of_oop_eligible_date_log'] = np.log1p(dt_prod_api_calc['osbal_as_of_oop_eligible_date'])

In [31]:
dt_prod_api_calc.head(2)

,ee_customer_id,run_date,attrition_risk_segment,attrition_time_to_leave,oop_score_prod,oop_risk_segment_prod,ee_onboarding_date,oop_target,score_oop,osbal_as_of_resignation_date,osbal_as_of_oop_eligible_date,osbal_as_of_current_date,onb_rd_diff,onboarding_date_ym,run_date_ym,osbal_as_of_oop_eligible_date_log
48438,1462050,2025-06-20,10-12,10,0.548142,B,2025-06-20,0,0.526408,17378.052,17378.052,17378.052,0.0,202506.0,202506,9.763021
134171,1462044,2025-06-21,9,9,0.559482,B,2025-06-21,1,0.539521,29230.912,29230.912,0.000,0.0,202506.0,202506,10.283016


In [32]:
print('OOP New Users Prod, weighted log')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Oct 2025': {'start': '2025-10-01', 'end': '2025-10-31'},
    'Nov 2025': {'start': '2025-11-01', 'end': '2025-11-30'},
    'Dec 2025': {'start': '2025-12-01', 'end': '2025-12-31'},
    'Jun 2025 - Dec 2025': {'start': '2025-06-01', 'end': '2025-12-31'}
}

calculate_gini_for_table(
    data = dt_prod_api_calc[dt_prod_api_calc['osbal_as_of_oop_eligible_date_log'].notna()],
    date_column = 'ee_onboarding_date',
    score_column = 'oop_score_prod',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col='osbal_as_of_oop_eligible_date_log'
)

OOP New Users Prod, weighted log

Gini Coefficient Results:
Jun 2025: 0.3725 (Sample size: 13)
Jul 2025: 0.0298 (Sample size: 84)
Aug 2025: 0.0697 (Sample size: 43)
Sep 2025: -0.505 (Sample size: 63)
Oct 2025: -0.0411 (Sample size: 24)
Nov 2025: 0.1365 (Sample size: 14)
Dec 2025: No data available for period 2025-12-01 to 2025-12-31
Jun 2025 - Dec 2025: -0.0838 (Sample size: 241)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30           13 61.538462            0.3725
           Jul 2025 2025-07-01 2025-07-31           84 71.428571            0.0298
           Aug 2025 2025-08-01 2025-08-31           43 76.744186            0.0697
           Sep 2025 2025-09-01 2025-09-30           63 76.190476           -0.5050
           Oct 2025 2025-10-01 2025-10-31           24 66.666667           -0.0411
           Nov 2025 2025-11-01 2025-11-30           14 78.571429            0.1365
           Dec 2025 

C:\Users\Dwaipayan\AppData\Local\Temp\ipykernel_61508\2145974915.py:52: RuntimeWarning: invalid value encountered in scalar divide
  bad_rate = 100*(1 - period_data[['ee_customer_id',target_column]].drop_duplicates()[target_column].sum() / sample_size)


,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,13,61.538462,0.3725
1,Jul 2025,2025-07-01,2025-07-31,84,71.428571,0.0298
2,Aug 2025,2025-08-01,2025-08-31,43,76.744186,0.0697
3,Sep 2025,2025-09-01,2025-09-30,63,76.190476,-0.5050
4,Oct 2025,2025-10-01,2025-10-31,24,66.666667,-0.0411
5,Nov 2025,2025-11-01,2025-11-30,14,78.571429,0.1365
6,Dec 2025,2025-12-01,2025-12-31,0,NaN,NaN
7,Jun 2025 - Dec 2025,2025-06-01,2025-12-31,241,73.029046,-0.0838


In [33]:
print('OOP New Users Prod BS, weighted as-is')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Jun 2025 - Sep 2025': {'start': '2025-06-01', 'end': '2025-09-30'}
}

calculate_gini_for_table(
    data = dt_prod_api_calc[(dt_prod_api_calc['score_oop'].notna()) & dt_prod_api_calc['osbal_as_of_oop_eligible_date'].notna()],
    date_column = 'ee_onboarding_date',
    score_column = 'score_oop',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col='osbal_as_of_oop_eligible_date'
)

OOP New Users Prod BS, weighted as-is

Gini Coefficient Results:
Jun 2025: 0.3115 (Sample size: 13)
Jul 2025: 0.2418 (Sample size: 84)
Aug 2025: -0.1151 (Sample size: 43)
Sep 2025: -0.4345 (Sample size: 63)
Jun 2025 - Sep 2025: -0.0153 (Sample size: 203)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30           13 61.538462            0.3115
           Jul 2025 2025-07-01 2025-07-31           84 71.428571            0.2418
           Aug 2025 2025-08-01 2025-08-31           43 76.744186           -0.1151
           Sep 2025 2025-09-01 2025-09-30           63 76.190476           -0.4345
Jun 2025 - Sep 2025 2025-06-01 2025-09-30          203 73.399015           -0.0153


,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,13,61.538462,0.3115
1,Jul 2025,2025-07-01,2025-07-31,84,71.428571,0.2418
2,Aug 2025,2025-08-01,2025-08-31,43,76.744186,-0.1151
3,Sep 2025,2025-09-01,2025-09-30,63,76.190476,-0.4345
4,Jun 2025 - Sep 2025,2025-06-01,2025-09-30,203,73.399015,-0.0153


In [34]:
print('OOP New Users Prod BS, weighted log transformed')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Oct 2025': {'start': '2025-10-01', 'end': '2025-10-31'},
    'Nov 2025': {'start': '2025-11-01', 'end': '2025-11-30'},
    'Dec 2025': {'start': '2025-12-01', 'end': '2025-12-31'},
    'Jun 2025 - Dec 2025': {'start': '2025-06-01', 'end': '2025-12-31'}
}

calculate_gini_for_table(
    data = dt_prod_api_calc[(dt_prod_api_calc['score_oop'].notna()) & dt_prod_api_calc['osbal_as_of_oop_eligible_date_log'].notna()],
    date_column = 'ee_onboarding_date',
    score_column = 'score_oop',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col='osbal_as_of_oop_eligible_date_log'
)

OOP New Users Prod BS, weighted log transformed

Gini Coefficient Results:
Jun 2025: 0.1636 (Sample size: 13)
Jul 2025: 0.3009 (Sample size: 84)
Aug 2025: 0.1705 (Sample size: 43)
Sep 2025: -0.3031 (Sample size: 63)
Oct 2025: 0.0987 (Sample size: 24)
Nov 2025: 0.1365 (Sample size: 14)
Dec 2025: No data available for period 2025-12-01 to 2025-12-31
Jun 2025 - Dec 2025: 0.0738 (Sample size: 241)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30           13 61.538462            0.1636
           Jul 2025 2025-07-01 2025-07-31           84 71.428571            0.3009
           Aug 2025 2025-08-01 2025-08-31           43 76.744186            0.1705
           Sep 2025 2025-09-01 2025-09-30           63 76.190476           -0.3031
           Oct 2025 2025-10-01 2025-10-31           24 66.666667            0.0987
           Nov 2025 2025-11-01 2025-11-30           14 78.571429            0.1365
      

C:\Users\Dwaipayan\AppData\Local\Temp\ipykernel_61508\2145974915.py:52: RuntimeWarning: invalid value encountered in scalar divide
  bad_rate = 100*(1 - period_data[['ee_customer_id',target_column]].drop_duplicates()[target_column].sum() / sample_size)


,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,13,61.538462,0.1636
1,Jul 2025,2025-07-01,2025-07-31,84,71.428571,0.3009
2,Aug 2025,2025-08-01,2025-08-31,43,76.744186,0.1705
3,Sep 2025,2025-09-01,2025-09-30,63,76.190476,-0.3031
4,Oct 2025,2025-10-01,2025-10-31,24,66.666667,0.0987
5,Nov 2025,2025-11-01,2025-11-30,14,78.571429,0.1365
6,Dec 2025,2025-12-01,2025-12-31,0,NaN,NaN
7,Jun 2025 - Dec 2025,2025-06-01,2025-12-31,241,73.029046,0.0738


### prod existing

In [35]:
dt_prod_batch.shape

(385689, 6)

In [36]:
dt_prod_batch['ee_customer_id'].nunique()

78520

In [37]:
dt_prod_batch = dt_prod_batch.drop_duplicates(subset=['ee_customer_id','run_date','attrition_time_to_leave','oop_score_prod'])

In [38]:
dt_prod_batch.shape

(384214, 6)

In [39]:
dt_prod_batch["run_date"] = pd.to_datetime(dt_prod_batch["run_date"], errors="coerce")
dt_prod_batch['run_date_ym'] = dt_prod_batch['run_date'].dt.year*100 + dt_prod_batch['run_date'].dt.month

In [40]:
dt_prod_batch.head(2)

,ee_customer_id,run_date,attrition_risk_segment,attrition_time_to_leave,oop_score_prod,oop_risk_segment_prod,run_date_ym
0,1157338,2025-10-14,Low,10,0.704469,A,202510
1,1456942,2025-10-14,Low,10,0.735589,A,202510


In [41]:
dt_bs_oop_ex.tail(2)

,ee_customer_id,calc_date,target_dev,score_oop,calc_date_correct,calc_date_ym
1427945,999581,2026-01-01,NaN,0.691222,2025-12-01,202512
1427946,999921,2026-01-01,NaN,0.453340,2025-12-01,202512


In [42]:
dt_prod_batch = dt_prod_batch.merge(
    dt_raw[['ee_customer_id','ee_onboarding_date']].drop_duplicates(),
    how='left',
    on='ee_customer_id'
).merge(
    dt_oop_targets,
    how='left',
    on='ee_customer_id'
).merge(
    dt_bs_oop_new[['ee_customer_id', 'osbal_as_of_resignation_date', 'osbal_as_of_oop_eligible_date', 'osbal_as_of_current_date']],
    how='left',
    on='ee_customer_id'
).merge(
    dt_bs_oop_ex[['ee_customer_id', 'calc_date_ym', 'score_oop']],
    how='left',
    left_on=['ee_customer_id','run_date_ym'],
    right_on=['ee_customer_id','calc_date_ym']
)

In [43]:
dt_prod_batch.columns

Index(['ee_customer_id', 'run_date', 'attrition_risk_segment',
       'attrition_time_to_leave', 'oop_score_prod', 'oop_risk_segment_prod',
       'run_date_ym', 'ee_onboarding_date', 'oop_target',
       'osbal_as_of_resignation_date', 'osbal_as_of_oop_eligible_date',
       'osbal_as_of_current_date', 'calc_date_ym', 'score_oop'],
      dtype='object')

In [44]:
dt_prod_batch.rename(columns={'ee_onboarding_date_x':'ee_onboarding_date', }, inplace=True)

In [45]:
dt_prod_batch['ee_onboarding_date'] = pd.to_datetime(dt_prod_batch['ee_onboarding_date']).dt.normalize()
dt_prod_batch['onboarding_date_ym'] = dt_prod_batch['ee_onboarding_date'].dt.year*100 + dt_prod_batch['ee_onboarding_date'].dt.month
dt_prod_batch['onb_rd_diff'] = (abs(dt_prod_batch['run_date'] - dt_prod_batch['ee_onboarding_date'])).dt.days

In [46]:
dt_prod_batch_calc = (dt_prod_batch
                    .dropna(subset=['ee_onboarding_date','oop_target'])
                    .sort_values(['ee_customer_id','run_date']))

In [47]:
dt_prod_batch_calc.head(1)

,ee_customer_id,run_date,attrition_risk_segment,attrition_time_to_leave,oop_score_prod,oop_risk_segment_prod,run_date_ym,ee_onboarding_date,oop_target,osbal_as_of_resignation_date,osbal_as_of_oop_eligible_date,osbal_as_of_current_date,calc_date_ym,score_oop,onboarding_date_ym,onb_rd_diff
300492,1002670,2025-06-08,High,5,0.549675,B,202506,2023-09-08,1,NaN,NaN,7339.019,202506.0,0.70453,202309.0,639.0


In [49]:
print('OOP Existing Users Prod')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Jun 2025 - Sep 2025': {'start': '2025-06-01', 'end': '2025-09-30'}
}

calculate_gini_for_table(
    data = dt_prod_batch_calc,
    date_column = 'run_date',
    score_column = 'oop_score_prod',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict
)

OOP Existing Users Prod

Gini Coefficient Results:
Jun 2025: 0.2572 (Sample size: 1,960)
Jul 2025: 0.2551 (Sample size: 3,111)
Aug 2025: 0.2385 (Sample size: 2,412)
Sep 2025: 0.2389 (Sample size: 1,789)
Jun 2025 - Sep 2025: 0.2464 (Sample size: 9,272)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30         1944 73.662551            0.2572
           Jul 2025 2025-07-01 2025-07-31         3111 71.263259            0.2551
           Aug 2025 2025-08-01 2025-08-31         2412 68.366501            0.2385
           Sep 2025 2025-09-01 2025-09-30         1789 65.064282            0.2389
Jun 2025 - Sep 2025 2025-06-01 2025-09-30         3999 71.942986            0.2464


,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,1944,73.662551,0.2572
1,Jul 2025,2025-07-01,2025-07-31,3111,71.263259,0.2551
2,Aug 2025,2025-08-01,2025-08-31,2412,68.366501,0.2385
3,Sep 2025,2025-09-01,2025-09-30,1789,65.064282,0.2389
4,Jun 2025 - Sep 2025,2025-06-01,2025-09-30,3999,71.942986,0.2464


In [50]:
print('OOP Existing Users Prod BS')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Jun 2025 - Sep 2025': {'start': '2025-06-01', 'end': '2025-09-30'}
}

calculate_gini_for_table(
    data = dt_prod_batch_calc[dt_prod_batch_calc['score_oop'].notna()],
    date_column = 'run_date',
    score_column = 'score_oop',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict
)

OOP Existing Users Prod BS

Gini Coefficient Results:
Jun 2025: 0.2849 (Sample size: 1,420)
Jul 2025: 0.2715 (Sample size: 2,141)
Aug 2025: 0.2769 (Sample size: 1,867)
Sep 2025: 0.2982 (Sample size: 1,380)
Jun 2025 - Sep 2025: 0.2806 (Sample size: 6,808)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30         1406 73.470839            0.2849
           Jul 2025 2025-07-01 2025-07-31         2141 69.406819            0.2715
           Aug 2025 2025-08-01 2025-08-31         1867 67.487949            0.2769
           Sep 2025 2025-09-01 2025-09-30         1380 64.275362            0.2982
Jun 2025 - Sep 2025 2025-06-01 2025-09-30         2844 71.343179            0.2806


,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,1406,73.470839,0.2849
1,Jul 2025,2025-07-01,2025-07-31,2141,69.406819,0.2715
2,Aug 2025,2025-08-01,2025-08-31,1867,67.487949,0.2769
3,Sep 2025,2025-09-01,2025-09-30,1380,64.275362,0.2982
4,Jun 2025 - Sep 2025,2025-06-01,2025-09-30,2844,71.343179,0.2806


In [51]:
print('OOP Existing Users Prod, weighted as-is')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Jun 2025 - Sep 2025': {'start': '2025-06-01', 'end': '2025-09-30'}
}

calculate_gini_for_table(
    data = dt_prod_batch_calc[dt_prod_batch_calc['osbal_as_of_oop_eligible_date'].notna()],
    date_column = 'run_date',
    score_column = 'oop_score_prod',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col = 'osbal_as_of_oop_eligible_date'
)

OOP Existing Users Prod, weighted as-is

Gini Coefficient Results:
Jun 2025: 0.261 (Sample size: 1,593)
Jul 2025: 0.2114 (Sample size: 2,418)
Aug 2025: 0.2153 (Sample size: 1,786)
Sep 2025: 0.2115 (Sample size: 1,209)
Jun 2025 - Sep 2025: 0.2211 (Sample size: 7,006)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30         1584 78.661616            0.2610
           Jul 2025 2025-07-01 2025-07-31         2418 77.460711            0.2114
           Aug 2025 2025-08-01 2025-08-31         1786 76.595745            0.2153
           Sep 2025 2025-09-01 2025-09-30         1209 76.013234            0.2115
Jun 2025 - Sep 2025 2025-06-01 2025-09-30         3140 77.802548            0.2211


,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,1584,78.661616,0.2610
1,Jul 2025,2025-07-01,2025-07-31,2418,77.460711,0.2114
2,Aug 2025,2025-08-01,2025-08-31,1786,76.595745,0.2153
3,Sep 2025,2025-09-01,2025-09-30,1209,76.013234,0.2115
4,Jun 2025 - Sep 2025,2025-06-01,2025-09-30,3140,77.802548,0.2211


In [52]:
dt_prod_batch_calc['osbal_as_of_oop_eligible_date_log'] = np.log1p(dt_prod_batch_calc['osbal_as_of_oop_eligible_date'])

In [53]:
print('OOP Existing Users Prod, weighted log transformed')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Oct 2025': {'start': '2025-10-01', 'end': '2025-10-31'},
    'Nov 2025': {'start': '2025-11-01', 'end': '2025-11-30'},
    'Dec 2025': {'start': '2025-12-01', 'end': '2025-12-31'},
    'Jun 2025 - Dec 2025': {'start': '2025-06-01', 'end': '2025-12-31'}
}

calculate_gini_for_table(
    data = dt_prod_batch_calc[dt_prod_batch_calc['osbal_as_of_oop_eligible_date_log'].notna()],
    date_column = 'run_date',
    score_column = 'oop_score_prod',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col = 'osbal_as_of_oop_eligible_date_log'
)

OOP Existing Users Prod, weighted log transformed

Gini Coefficient Results:
Jun 2025: 0.2673 (Sample size: 1,593)
Jul 2025: 0.2756 (Sample size: 2,418)
Aug 2025: 0.282 (Sample size: 1,786)
Sep 2025: 0.3006 (Sample size: 1,209)
Oct 2025: 0.2517 (Sample size: 864)
Nov 2025: 0.1348 (Sample size: 548)
Dec 2025: 0.162 (Sample size: 224)
Jun 2025 - Dec 2025: 0.2539 (Sample size: 8,642)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30         1584 78.661616            0.2673
           Jul 2025 2025-07-01 2025-07-31         2418 77.460711            0.2756
           Aug 2025 2025-08-01 2025-08-31         1786 76.595745            0.2820
           Sep 2025 2025-09-01 2025-09-30         1209 76.013234            0.3006
           Oct 2025 2025-10-01 2025-10-31          864 74.189815            0.2517
           Nov 2025 2025-11-01 2025-11-30          548 74.635036            0.1348
           Dec 2025

,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,1584,78.661616,0.2673
1,Jul 2025,2025-07-01,2025-07-31,2418,77.460711,0.2756
2,Aug 2025,2025-08-01,2025-08-31,1786,76.595745,0.2820
3,Sep 2025,2025-09-01,2025-09-30,1209,76.013234,0.3006
4,Oct 2025,2025-10-01,2025-10-31,864,74.189815,0.2517
5,Nov 2025,2025-11-01,2025-11-30,548,74.635036,0.1348
6,Dec 2025,2025-12-01,2025-12-31,224,75.446429,0.1620
7,Jun 2025 - Dec 2025,2025-06-01,2025-12-31,3357,77.539470,0.2539


In [54]:
print('OOP Existing Users Prod BS, weighted as-is')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Jun 2025 - Sep 2025': {'start': '2025-06-01', 'end': '2025-09-30'}
}

calculate_gini_for_table(
    data = dt_prod_batch_calc[(dt_prod_batch_calc['score_oop'].notna()) & (dt_prod_batch_calc['osbal_as_of_oop_eligible_date'].notna())],
    date_column = 'run_date',
    score_column = 'score_oop',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col = 'osbal_as_of_oop_eligible_date'
)

OOP Existing Users Prod BS, weighted as-is

Gini Coefficient Results:
Jun 2025: 0.2703 (Sample size: 1,115)
Jul 2025: 0.1998 (Sample size: 1,550)
Aug 2025: 0.2167 (Sample size: 1,332)
Sep 2025: 0.2651 (Sample size: 892)
Jun 2025 - Sep 2025: 0.2298 (Sample size: 4,889)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30         1107 79.855465            0.2703
           Jul 2025 2025-07-01 2025-07-31         1550 77.806452            0.1998
           Aug 2025 2025-08-01 2025-08-31         1332 77.027027            0.2167
           Sep 2025 2025-09-01 2025-09-30          892 76.121076            0.2651
Jun 2025 - Sep 2025 2025-06-01 2025-09-30         2139 78.681627            0.2298


,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,1107,79.855465,0.2703
1,Jul 2025,2025-07-01,2025-07-31,1550,77.806452,0.1998
2,Aug 2025,2025-08-01,2025-08-31,1332,77.027027,0.2167
3,Sep 2025,2025-09-01,2025-09-30,892,76.121076,0.2651
4,Jun 2025 - Sep 2025,2025-06-01,2025-09-30,2139,78.681627,0.2298


In [55]:
print('OOP Existing Users Prod BSm weighted log transformed')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Oct 2025': {'start': '2025-10-01', 'end': '2025-10-31'},
    'Nov 2025': {'start': '2025-11-01', 'end': '2025-11-30'},
    'Dec 2025': {'start': '2025-12-01', 'end': '2025-12-31'},
    'Jun 2025 - Dec 2025': {'start': '2025-06-01', 'end': '2025-12-31'}
}

calculate_gini_for_table(
    data = dt_prod_batch_calc[(dt_prod_batch_calc['score_oop'].notna()) & (dt_prod_batch_calc['osbal_as_of_oop_eligible_date_log'].notna())],
    date_column = 'run_date',
    score_column = 'score_oop',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col = 'osbal_as_of_oop_eligible_date_log'
)

OOP Existing Users Prod BSm weighted log transformed

Gini Coefficient Results:
Jun 2025: 0.2808 (Sample size: 1,115)
Jul 2025: 0.2881 (Sample size: 1,550)
Aug 2025: 0.3054 (Sample size: 1,332)
Sep 2025: 0.3453 (Sample size: 892)
Oct 2025: 0.3194 (Sample size: 640)
Nov 2025: 0.2529 (Sample size: 318)
Dec 2025: 0.3925 (Sample size: 43)
Jun 2025 - Dec 2025: 0.3027 (Sample size: 5,890)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30         1107 79.855465            0.2808
           Jul 2025 2025-07-01 2025-07-31         1550 77.806452            0.2881
           Aug 2025 2025-08-01 2025-08-31         1332 77.027027            0.3054
           Sep 2025 2025-09-01 2025-09-30          892 76.121076            0.3453
           Oct 2025 2025-10-01 2025-10-31          640 74.375000            0.3194
           Nov 2025 2025-11-01 2025-11-30          318 77.044025            0.2529
           Dec 20

,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,1107,79.855465,0.2808
1,Jul 2025,2025-07-01,2025-07-31,1550,77.806452,0.2881
2,Aug 2025,2025-08-01,2025-08-31,1332,77.027027,0.3054
3,Sep 2025,2025-09-01,2025-09-30,892,76.121076,0.3453
4,Oct 2025,2025-10-01,2025-10-31,640,74.375000,0.3194
5,Nov 2025,2025-11-01,2025-11-30,318,77.044025,0.2529
6,Dec 2025,2025-12-01,2025-12-31,43,69.767442,0.3925
7,Jun 2025 - Dec 2025,2025-06-01,2025-12-31,2260,78.451327,0.3027


### bs new

In [59]:
dt_bs_oop_new.head(1)

,ee_customer_id,calc_date,target_dev,score_oop,osbal_as_of_resignation_date,osbal_as_of_oop_eligible_date,osbal_as_of_current_date
0,1211049,2024-07-01,NaN,0.104421,NaN,NaN,0.003


In [58]:
dt_bs_oop_new.shape

(162644, 7)

In [60]:
dt_bs_oop_new['ee_customer_id'].nunique()

162644

In [61]:
dt_bs_oop_new = dt_bs_oop_new.merge(
    dt_raw[['ee_customer_id','ee_onboarding_date']].drop_duplicates(),
    how='left',
    on='ee_customer_id'
).merge(
    dt_oop_targets,
    how='left',
    on='ee_customer_id'
)

In [62]:
dt_bs_oop_new['ee_onboarding_date'] = pd.to_datetime(dt_bs_oop_new['ee_onboarding_date']).dt.normalize()
dt_bs_oop_new['onboarding_date_ym'] = dt_bs_oop_new['ee_onboarding_date'].dt.year*100 + dt_bs_oop_new['ee_onboarding_date'].dt.month

In [63]:
dt_bs_oop_new_calc = dt_bs_oop_new.dropna(subset=['oop_target'])

In [64]:
dt_bs_oop_new_calc.head(1)

,ee_customer_id,calc_date,target_dev,score_oop,osbal_as_of_resignation_date,osbal_as_of_oop_eligible_date,osbal_as_of_current_date,ee_onboarding_date,oop_target,onboarding_date_ym
34,1206189,2024-07-01,NaN,0.126049,29162.768,29162.768,29162.768,2024-06-05,0,202406


In [65]:
print('OOP New Users BS')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Jun 2025 - Sep 2025': {'start': '2025-06-01', 'end': '2025-09-30'}
}

calculate_gini_for_table(
    data = dt_bs_oop_new_calc,
    date_column = 'ee_onboarding_date',
    score_column = 'score_oop',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict
)

OOP New Users BS

Gini Coefficient Results:
Jun 2025: 0.1028 (Sample size: 408)
Jul 2025: 0.1088 (Sample size: 389)
Aug 2025: 0.079 (Sample size: 220)
Sep 2025: -0.0263 (Sample size: 525)
Jun 2025 - Sep 2025: 0.0581 (Sample size: 1,542)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30          408 65.686275            0.1028
           Jul 2025 2025-07-01 2025-07-31          389 59.640103            0.1088
           Aug 2025 2025-08-01 2025-08-31          220 55.000000            0.0790
           Sep 2025 2025-09-01 2025-09-30          525 62.857143           -0.0263
Jun 2025 - Sep 2025 2025-06-01 2025-09-30         1542 61.673152            0.0581


,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,408,65.686275,0.1028
1,Jul 2025,2025-07-01,2025-07-31,389,59.640103,0.1088
2,Aug 2025,2025-08-01,2025-08-31,220,55.000000,0.0790
3,Sep 2025,2025-09-01,2025-09-30,525,62.857143,-0.0263
4,Jun 2025 - Sep 2025,2025-06-01,2025-09-30,1542,61.673152,0.0581


In [66]:
dt_bs_oop_new_calc['osbal_as_of_oop_eligible_date_log'] = np.log1p(dt_bs_oop_new_calc['osbal_as_of_oop_eligible_date'])

C:\Users\Dwaipayan\AppData\Local\Temp\ipykernel_61508\3448610563.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dt_bs_oop_new_calc['osbal_as_of_oop_eligible_date_log'] = np.log1p(dt_bs_oop_new_calc['osbal_as_of_oop_eligible_date'])


In [68]:
print('OOP Existing Users BS, weighted as-is')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Jun 2025 - Sep 2025': {'start': '2025-06-01', 'end': '2025-09-30'}
}

calculate_gini_for_table(
    data = dt_bs_oop_new_calc[dt_bs_oop_new_calc['osbal_as_of_oop_eligible_date'].notna()],
    date_column = 'ee_onboarding_date',
    score_column = 'score_oop',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col = 'osbal_as_of_oop_eligible_date'
)

OOP Existing Users BS, weighted as-is

Gini Coefficient Results:
Jun 2025: 0.0668 (Sample size: 349)
Jul 2025: 0.048 (Sample size: 296)
Aug 2025: 0.0138 (Sample size: 147)
Sep 2025: 0.0041 (Sample size: 382)
Jun 2025 - Sep 2025: 0.0207 (Sample size: 1,174)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30          349 72.492837            0.0668
           Jul 2025 2025-07-01 2025-07-31          296 70.945946            0.0480
           Aug 2025 2025-08-01 2025-08-31          147 68.707483            0.0138
           Sep 2025 2025-09-01 2025-09-30          382 75.916230            0.0041
Jun 2025 - Sep 2025 2025-06-01 2025-09-30         1174 72.742760            0.0207


,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,349,72.492837,0.0668
1,Jul 2025,2025-07-01,2025-07-31,296,70.945946,0.0480
2,Aug 2025,2025-08-01,2025-08-31,147,68.707483,0.0138
3,Sep 2025,2025-09-01,2025-09-30,382,75.916230,0.0041
4,Jun 2025 - Sep 2025,2025-06-01,2025-09-30,1174,72.742760,0.0207


In [69]:
print('OOP Existing Users BS, weighted log')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Oct 2025': {'start': '2025-10-01', 'end': '2025-10-31'},
    'Nov 2025': {'start': '2025-11-01', 'end': '2025-11-30'},
    'Dec 2025': {'start': '2025-12-01', 'end': '2025-12-31'},
    'Jun 2025 - Dec 2025': {'start': '2025-06-01', 'end': '2025-12-31'}
}

calculate_gini_for_table(
    data = dt_bs_oop_new_calc[dt_bs_oop_new_calc['osbal_as_of_oop_eligible_date_log'].notna()],
    date_column = 'ee_onboarding_date',
    score_column = 'score_oop',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col = 'osbal_as_of_oop_eligible_date_log'
)

OOP Existing Users BS, weighted log

Gini Coefficient Results:
Jun 2025: 0.1076 (Sample size: 349)
Jul 2025: 0.0535 (Sample size: 296)
Aug 2025: 0.0719 (Sample size: 147)
Sep 2025: -0.0142 (Sample size: 382)
Oct 2025: 0.2153 (Sample size: 448)
Nov 2025: 0.2419 (Sample size: 163)
Dec 2025: -0.754 (Sample size: 6)
Jun 2025 - Dec 2025: 0.114 (Sample size: 1,791)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30          349 72.492837            0.1076
           Jul 2025 2025-07-01 2025-07-31          296 70.945946            0.0535
           Aug 2025 2025-08-01 2025-08-31          147 68.707483            0.0719
           Sep 2025 2025-09-01 2025-09-30          382 75.916230           -0.0142
           Oct 2025 2025-10-01 2025-10-31          448 80.803571            0.2153
           Nov 2025 2025-11-01 2025-11-30          163 82.822086            0.2419
           Dec 2025 2025-12-01 2025-12-31

,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,349,72.492837,0.1076
1,Jul 2025,2025-07-01,2025-07-31,296,70.945946,0.0535
2,Aug 2025,2025-08-01,2025-08-31,147,68.707483,0.0719
3,Sep 2025,2025-09-01,2025-09-30,382,75.916230,-0.0142
4,Oct 2025,2025-10-01,2025-10-31,448,80.803571,0.2153
5,Nov 2025,2025-11-01,2025-11-30,163,82.822086,0.2419
6,Dec 2025,2025-12-01,2025-12-31,6,66.666667,-0.7540
7,Jun 2025 - Dec 2025,2025-06-01,2025-12-31,1791,75.656058,0.1140


### bs existing

In [70]:
dt_bs_oop_ex.head(1)

,ee_customer_id,calc_date,target_dev,score_oop,calc_date_correct,calc_date_ym
0,102999,2023-01-01,NaN,0.546834,2022-12-01,202212


In [71]:
dt_bs_oop_ex.shape

(1427947, 6)

In [72]:
dt_bs_oop_ex['ee_customer_id'].nunique()

132907

In [73]:
dt_bs_oop_ex = dt_bs_oop_ex.merge(
    dt_raw[['ee_customer_id','ee_onboarding_date']].drop_duplicates(),
    how='left',
    on='ee_customer_id'
).merge(
    dt_oop_targets,
    how='left',
    on='ee_customer_id'
).merge(
    dt_bs_oop_new[['ee_customer_id', 'osbal_as_of_resignation_date', 'osbal_as_of_oop_eligible_date', 'osbal_as_of_current_date']],
    how='left',
    on='ee_customer_id'
)

In [77]:
dt_bs_oop_ex.shape

(1427947, 11)

In [75]:
dt_bs_oop_ex.head(1)

,ee_customer_id,calc_date,target_dev,score_oop,calc_date_correct,calc_date_ym,ee_onboarding_date,oop_target,osbal_as_of_resignation_date,osbal_as_of_oop_eligible_date,osbal_as_of_current_date
0,102999,2023-01-01,NaN,0.546834,2022-12-01,202212,2022-07-31 17:18:28,<NA>,NaN,NaN,NaN


In [76]:
dt_bs_oop_ex_calc = dt_bs_oop_ex.dropna(subset=['oop_target'])

In [78]:
dt_bs_oop_ex_calc.head(1)

,ee_customer_id,calc_date,target_dev,score_oop,calc_date_correct,calc_date_ym,ee_onboarding_date,oop_target,osbal_as_of_resignation_date,osbal_as_of_oop_eligible_date,osbal_as_of_current_date
10,122607,2023-01-01,0.0,0.458975,2022-12-01,202212,2022-08-16 10:42:46,0,NaN,NaN,NaN


In [79]:
print('OOP Existing Users BS')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Jun 2025 - Sep 2025': {'start': '2025-06-01', 'end': '2025-09-30'}
}

calculate_gini_for_table(
    data = dt_bs_oop_ex_calc,
    date_column = 'calc_date_correct',
    score_column = 'score_oop',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict
)

OOP Existing Users BS

Gini Coefficient Results:
Jun 2025: 0.2626 (Sample size: 4,270)
Jul 2025: 0.2709 (Sample size: 4,638)
Aug 2025: 0.284 (Sample size: 4,215)
Sep 2025: 0.2831 (Sample size: 3,299)
Jun 2025 - Sep 2025: 0.2823 (Sample size: 16,422)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30         4270 76.580796            0.2626
           Jul 2025 2025-07-01 2025-07-31         4638 71.129797            0.2709
           Aug 2025 2025-08-01 2025-08-31         4215 69.062871            0.2840
           Sep 2025 2025-09-01 2025-09-30         3299 65.474386            0.2831
Jun 2025 - Sep 2025 2025-06-01 2025-09-30         6198 71.926428            0.2823


,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,4270,76.580796,0.2626
1,Jul 2025,2025-07-01,2025-07-31,4638,71.129797,0.2709
2,Aug 2025,2025-08-01,2025-08-31,4215,69.062871,0.2840
3,Sep 2025,2025-09-01,2025-09-30,3299,65.474386,0.2831
4,Jun 2025 - Sep 2025,2025-06-01,2025-09-30,6198,71.926428,0.2823


In [80]:
dt_bs_oop_ex_calc['osbal_as_of_oop_eligible_date_log'] = np.log1p(dt_bs_oop_ex_calc['osbal_as_of_oop_eligible_date'])

C:\Users\Dwaipayan\AppData\Local\Temp\ipykernel_61508\3634989807.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dt_bs_oop_ex_calc['osbal_as_of_oop_eligible_date_log'] = np.log1p(dt_bs_oop_ex_calc['osbal_as_of_oop_eligible_date'])


In [81]:
print('OOP Existing Users BS, weighted as-is')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Jun 2025 - Sep 2025': {'start': '2025-06-01', 'end': '2025-09-30'}
}

calculate_gini_for_table(
    data = dt_bs_oop_ex_calc[dt_bs_oop_ex_calc['osbal_as_of_oop_eligible_date'].notna()],
    date_column = 'calc_date_correct',
    score_column = 'score_oop',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col = 'osbal_as_of_oop_eligible_date'
)

OOP Existing Users BS, weighted as-is

Gini Coefficient Results:
Jun 2025: 0.2544 (Sample size: 3,129)
Jul 2025: 0.2359 (Sample size: 3,375)
Aug 2025: 0.2703 (Sample size: 3,002)
Sep 2025: 0.2793 (Sample size: 2,233)
Jun 2025 - Sep 2025: 0.2617 (Sample size: 11,739)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30         3129 81.335890            0.2544
           Jul 2025 2025-07-01 2025-07-31         3375 77.925926            0.2359
           Aug 2025 2025-08-01 2025-08-31         3002 77.548301            0.2703
           Sep 2025 2025-09-01 2025-09-30         2233 76.668159            0.2793
Jun 2025 - Sep 2025 2025-06-01 2025-09-30         4701 78.089768            0.2617


,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,3129,81.335890,0.2544
1,Jul 2025,2025-07-01,2025-07-31,3375,77.925926,0.2359
2,Aug 2025,2025-08-01,2025-08-31,3002,77.548301,0.2703
3,Sep 2025,2025-09-01,2025-09-30,2233,76.668159,0.2793
4,Jun 2025 - Sep 2025,2025-06-01,2025-09-30,4701,78.089768,0.2617


In [82]:
print('OOP Existing Users BS, weighted log transform')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Oct 2025': {'start': '2025-10-01', 'end': '2025-10-31'},
    'Nov 2025': {'start': '2025-11-01', 'end': '2025-11-30'},
    'Dec 2025': {'start': '2025-12-01', 'end': '2025-12-31'},
    'Jun 2025 - Dec 2025': {'start': '2025-06-01', 'end': '2025-12-31'}
}

calculate_gini_for_table(
    data = dt_bs_oop_ex_calc[dt_bs_oop_ex_calc['osbal_as_of_oop_eligible_date_log'].notna()],
    date_column = 'calc_date_correct',
    score_column = 'score_oop',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col = 'osbal_as_of_oop_eligible_date_log'
)

OOP Existing Users BS, weighted log transform

Gini Coefficient Results:
Jun 2025: 0.2815 (Sample size: 3,129)
Jul 2025: 0.2631 (Sample size: 3,375)
Aug 2025: 0.2762 (Sample size: 3,002)
Sep 2025: 0.2708 (Sample size: 2,233)
Oct 2025: 0.2348 (Sample size: 1,659)
Nov 2025: 0.2621 (Sample size: 958)
Dec 2025: 0.2523 (Sample size: 199)
Jun 2025 - Dec 2025: 0.272 (Sample size: 14,555)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30         3129 81.335890            0.2815
           Jul 2025 2025-07-01 2025-07-31         3375 77.925926            0.2631
           Aug 2025 2025-08-01 2025-08-31         3002 77.548301            0.2762
           Sep 2025 2025-09-01 2025-09-30         2233 76.668159            0.2708
           Oct 2025 2025-10-01 2025-10-31         1659 74.743822            0.2348
           Nov 2025 2025-11-01 2025-11-30          958 73.695198            0.2621
           Dec 2025

,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,3129,81.335890,0.2815
1,Jul 2025,2025-07-01,2025-07-31,3375,77.925926,0.2631
2,Aug 2025,2025-08-01,2025-08-31,3002,77.548301,0.2762
3,Sep 2025,2025-09-01,2025-09-30,2233,76.668159,0.2708
4,Oct 2025,2025-10-01,2025-10-31,1659,74.743822,0.2348
5,Nov 2025,2025-11-01,2025-11-30,958,73.695198,0.2621
6,Dec 2025,2025-12-01,2025-12-31,199,76.381910,0.2523
7,Jun 2025 - Dec 2025,2025-06-01,2025-12-31,4976,77.652733,0.2720


## Distribution metrics

### prod new

In [84]:
# finding oop score ranges
# 1) observed min/max per segment from production
mm = (
    dt_prod_api.query('run_date >= "2025-10-15"')
      .assign(
          oop_risk_segment_prod=lambda d: d["oop_risk_segment_prod"].astype(str),
          oop_score_prod=lambda d: pd.to_numeric(d["oop_score_prod"], errors="coerce"),
      )
      .pivot_table(index="oop_risk_segment_prod", values="oop_score_prod", aggfunc=["min", "max"])
)

# flatten columns: ('min','oop_score_prod') -> 'min', same for 'max'
mm = mm.droplevel(1, axis=1).rename(columns={"min": "min_score", "max": "max_score"})

# enforce segment order: A highest scores ... F lowest
order = list("ABCDEF")
mm = mm.reindex(order)

# clamp to [0,1] (safe even if scores are slightly outside)
mm["min_score"] = mm["min_score"].clip(0, 1)
mm["max_score"] = mm["max_score"].clip(0, 1)

# if any segments missing in prod sample, fill min/max by interpolation between neighbors
mm[["min_score", "max_score"]] = mm[["min_score", "max_score"]].interpolate(limit_direction="both")

# 2) compute cutpoints between adjacent segments:
# boundary between (A,B) = midpoint between min(A) and max(B), etc.
cut_idx = [f"{hi}/{lo}" for hi, lo in zip(order[:-1], order[1:])]
cuts = pd.Series(
    [(mm.loc[hi, "min_score"] + mm.loc[lo, "max_score"]) / 2 for hi, lo in zip(order[:-1], order[1:])],
    index=cut_idx,
)

# enforce monotonicity (A/B cutoff should be >= B/C cutoff >= ... >= E/F cutoff)
cuts = cuts.cummin()

# 3) build bins for pd.cut (ascending edges) + labels (F..A)
# edges: [0, cut_EF, cut_DE, ..., cut_AB, 1]
edges_new = [0.0] + cuts.iloc[::-1].tolist() + [np.nextafter(1.0, 2.0)]
labels_new = list("FEDCBA")

# Optional: a readable cutoff table
cutoff_table = pd.DataFrame({
    "segment": labels_new,
    "min_inclusive": edges_new[:-1],
    "max_exclusive": edges_new[1:],
})
cutoff_table.loc[cutoff_table["segment"] == "A", "max_exclusive"] = 1.0

In [85]:
dt_prod_api_calc["oop_risk_segment_bs"] = pd.cut(
    dt_prod_api_calc["score_oop"],
    bins=edges_new,
    labels=labels_new,
    right=False,          # intervals are [a, b) so boundary goes to the higher segment
    include_lowest=True,
)

In [86]:
# New prod users, prod score, June-Aug
df = dt_prod_api_calc.query('onboarding_date_ym >= 202506 & onboarding_date_ym <= 202508').copy()

df["oop_target_bad"] = 1 - df["oop_target"]

df["osbal_current_bad"] = df["oop_target_bad"] * df["osbal_as_of_current_date"]

pt = pd.pivot_table(
    df,
    index="oop_risk_segment_prod",
    values=["oop_target_bad", "osbal_as_of_resignation_date", "osbal_current_bad"],
    aggfunc={
        "oop_target_bad": ["count", "mean"],
        "osbal_as_of_resignation_date": "sum",
        "osbal_current_bad": "sum"
    }
)

pt[("php_weighted_outstanding_bad_rate", "ratio")] = (
    pt[("osbal_current_bad", "sum")] / pt[("osbal_as_of_resignation_date", "sum")]
)

pt

oop_target_bad           osbal_as_of_resignation_date  \
                               count      mean                          sum   
oop_risk_segment_prod                                                         
A                                 73  0.630137                   710310.722   
B                                 55  0.636364                   469654.730   
C                                 35  0.685714                   272508.969   
D                                  5       1.0                    38470.372   
E                                  8       0.5                    30225.107   
F                                 12       0.5                    52673.299   

                      osbal_current_bad php_weighted_outstanding_bad_rate  
                                    sum                             ratio  
oop_risk_segment_prod                                                      
A                            686442.525                          0.966398  
B                            342161.225                          0.728538  
C                            227656.567                          0.835409  
D                             23881.639                           0.62078  
E                             28135.287                          0.930858  
F                             51385.756                          0.975556

In [87]:
# New prod users, BS score, June-Aug
df = dt_prod_api_calc.query('onboarding_date_ym >= 202506 & onboarding_date_ym <= 202508').copy()

df["oop_target_bad"] = 1 - df["oop_target"]

df["osbal_current_bad"] = df["oop_target_bad"] * df["osbal_as_of_current_date"]

pt = pd.pivot_table(
    df,
    index="oop_risk_segment_bs",
    values=["oop_target_bad", "osbal_as_of_resignation_date", "osbal_current_bad"],
    aggfunc={
        "oop_target_bad": ["count", "mean"],
        "osbal_as_of_resignation_date": "sum",
        "osbal_current_bad": "sum"
    }
)

pt[("php_weighted_outstanding_bad_rate", "ratio")] = (
    pt[("osbal_current_bad", "sum")] / pt[("osbal_as_of_resignation_date", "sum")]
)

pt.sort_index(ascending=False)

oop_target_bad           osbal_as_of_resignation_date  \
                             count      mean                          sum   
oop_risk_segment_bs                                                         
A                                4      0.75                    55829.887   
B                               67  0.567164                   627970.897   
C                               63  0.666667                   613626.747   
D                               24  0.666667                   174291.917   
E                               13  0.538462                    61579.421   
F                                7       1.0                    40544.330   

                    osbal_current_bad php_weighted_outstanding_bad_rate  
                                  sum                             ratio  
oop_risk_segment_bs                                                      
A                           34353.208                          0.615319  
B                          555226.792                           0.88416  
C                          517865.823                          0.843943  
D                          165192.923                          0.947795  
E                           46479.923                          0.754796  
F                            40544.33                               1.0

### bs new

In [88]:
dt_bs_oop_new_calc["oop_risk_segment_bs"] = pd.cut(
    dt_bs_oop_new_calc["score_oop"],
    bins=edges_new,
    labels=labels_new,
    right=False,          # intervals are [a, b) so boundary goes to the higher segment
    include_lowest=True,
)

C:\Users\Dwaipayan\AppData\Local\Temp\ipykernel_61508\2931598673.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dt_bs_oop_new_calc["oop_risk_segment_bs"] = pd.cut(


In [89]:
# New BS users, BS score, June-Aug
df = dt_bs_oop_new_calc.query('onboarding_date_ym >= 202506 & onboarding_date_ym <= 202508').copy()

df["oop_target_bad"] = 1 - df["oop_target"]

df["osbal_current_bad"] = df["oop_target_bad"] * df["osbal_as_of_current_date"]

pt = pd.pivot_table(
    df,
    index="oop_risk_segment_bs",
    values=["oop_target_bad", "osbal_as_of_resignation_date", "osbal_current_bad"],
    aggfunc={
        "oop_target_bad": ["count", "mean"],
        "osbal_as_of_resignation_date": "sum",
        "osbal_current_bad": "sum"
    }
)

pt[("php_weighted_outstanding_bad_rate", "ratio")] = (
    pt[("osbal_current_bad", "sum")] / pt[("osbal_as_of_resignation_date", "sum")]
)

pt.sort_index(ascending=False)

oop_target_bad           osbal_as_of_resignation_date  \
                             count      mean                          sum   
oop_risk_segment_bs                                                         
A                               23  0.478261                   269898.583   
B                              528  0.573864                  5201592.778   
C                              268  0.664179                  2959323.950   
D                              121  0.595041                  1392705.398   
E                               55  0.672727                   569892.585   
F                               22  0.909091                   230046.680   

                    osbal_current_bad php_weighted_outstanding_bad_rate  
                                  sum                             ratio  
oop_risk_segment_bs                                                      
A                          165750.024                           0.61412  
B                         3210664.393                          0.617246  
C                         2354997.396                          0.795789  
D                          808416.984                          0.580465  
E                          437205.939                          0.767173  
F                           214044.04                          0.930437

### prod existing

In [90]:
# finding oop score ranges
# 1) observed min/max per segment from production
mm = (
    dt_prod_batch.query('run_date >= "2025-10-15"')
      .assign(
          oop_risk_segment_prod=lambda d: d["oop_risk_segment_prod"].astype(str),
          oop_score_prod=lambda d: pd.to_numeric(d["oop_score_prod"], errors="coerce"),
      )
      .pivot_table(index="oop_risk_segment_prod", values="oop_score_prod", aggfunc=["min", "max"])
)

# flatten columns: ('min','oop_score_prod') -> 'min', same for 'max'
mm = mm.droplevel(1, axis=1).rename(columns={"min": "min_score", "max": "max_score"})

# enforce segment order: A highest scores ... F lowest
order = list("ABCDEF")
mm = mm.reindex(order)

# clamp to [0,1] (safe even if scores are slightly outside)
mm["min_score"] = mm["min_score"].clip(0, 1)
mm["max_score"] = mm["max_score"].clip(0, 1)

# if any segments missing in prod sample, fill min/max by interpolation between neighbors
mm[["min_score", "max_score"]] = mm[["min_score", "max_score"]].interpolate(limit_direction="both")

# 2) compute cutpoints between adjacent segments:
# boundary between (A,B) = midpoint between min(A) and max(B), etc.
cut_idx = [f"{hi}/{lo}" for hi, lo in zip(order[:-1], order[1:])]
cuts = pd.Series(
    [(mm.loc[hi, "min_score"] + mm.loc[lo, "max_score"]) / 2 for hi, lo in zip(order[:-1], order[1:])],
    index=cut_idx,
)

# enforce monotonicity (A/B cutoff should be >= B/C cutoff >= ... >= E/F cutoff)
cuts = cuts.cummin()

# 3) build bins for pd.cut (ascending edges) + labels (F..A)
# edges: [0, cut_EF, cut_DE, ..., cut_AB, 1]
edges_ex = [0.0] + cuts.iloc[::-1].tolist() + [np.nextafter(1.0, 2.0)]
labels_ex = list("FEDCBA")

# Optional: a readable cutoff table
cutoff_table = pd.DataFrame({
    "segment": labels_ex,
    "min_inclusive": edges_ex[:-1],
    "max_exclusive": edges_ex[1:],
})
cutoff_table.loc[cutoff_table["segment"] == "A", "max_exclusive"] = 1.0

In [91]:
dt_prod_batch_calc["oop_risk_segment_bs"] = pd.cut(
    dt_prod_batch_calc["score_oop"],
    bins=edges_ex,
    labels=labels_ex,
    right=False,          # intervals are [a, b) so boundary goes to the higher segment
    include_lowest=True,
)

In [92]:
# Existing prod users, prod score, June
df = dt_prod_batch_calc.query('run_date_ym == 202506').copy()

df["oop_target_bad"] = 1 - df["oop_target"]

df["osbal_current_bad"] = df["oop_target_bad"] * df["osbal_as_of_current_date"]

pt = pd.pivot_table(
    df,
    index="oop_risk_segment_prod",
    values=["oop_target_bad", "osbal_as_of_resignation_date", "osbal_current_bad"],
    aggfunc={
        "oop_target_bad": ["count", "mean"],
        "osbal_as_of_resignation_date": "sum",
        "osbal_current_bad": "sum"
    }
)

pt[("php_weighted_outstanding_bad_rate", "ratio")] = (
    pt[("osbal_current_bad", "sum")] / pt[("osbal_as_of_resignation_date", "sum")]
)

pt.sort_index(ascending=True)

oop_target_bad           osbal_as_of_resignation_date  \
                               count      mean                          sum   
oop_risk_segment_prod                                                         
A                                 71   0.56338                  1153580.412   
B                                451  0.631929                  7267894.794   
C                                625    0.7088                  8352635.001   
D                                470  0.829787                  6586783.514   
E                                177   0.79661                  2053734.069   
F                                166  0.855422                  1725422.485   

                      osbal_current_bad php_weighted_outstanding_bad_rate  
                                    sum                             ratio  
oop_risk_segment_prod                                                      
A                            673361.016                          0.583714  
B                           4798648.563                          0.660253  
C                           6507645.632                          0.779113  
D                           5545596.409                          0.841928  
E                           1671137.266                          0.813707  
F                           1558747.897                          0.903401

In [93]:
# Existing prod users, prod score, July
df = dt_prod_batch_calc.query('run_date_ym == 202507').copy()

df["oop_target_bad"] = 1 - df["oop_target"]

df["osbal_current_bad"] = df["oop_target_bad"] * df["osbal_as_of_current_date"]

pt = pd.pivot_table(
    df,
    index="oop_risk_segment_prod",
    values=["oop_target_bad", "osbal_as_of_resignation_date", "osbal_current_bad"],
    aggfunc={
        "oop_target_bad": ["count", "mean"],
        "osbal_as_of_resignation_date": "sum",
        "osbal_current_bad": "sum"
    }
)

pt[("php_weighted_outstanding_bad_rate", "ratio")] = (
    pt[("osbal_current_bad", "sum")] / pt[("osbal_as_of_resignation_date", "sum")]
)

pt.sort_index(ascending=True)

oop_target_bad           osbal_as_of_resignation_date  \
                               count      mean                          sum   
oop_risk_segment_prod                                                         
A                                102  0.529412                 1.477082e+06   
B                                716  0.610335                 1.103286e+07   
C                                962  0.685031                 1.438055e+07   
D                                765  0.781699                 1.108691e+07   
E                                304  0.802632                 3.740295e+06   
F                                262  0.858779                 2.703451e+06   

                      osbal_current_bad php_weighted_outstanding_bad_rate  
                                    sum                             ratio  
oop_risk_segment_prod                                                      
A                           1035281.743                          0.700896  
B                           8237272.353                          0.746612  
C                          10545283.711                          0.733302  
D                           9437250.067                          0.851206  
E                           3228647.269                          0.863207  
F                            2512607.39                          0.929407

In [94]:
# Existing prod users, prod score, Aug
df = dt_prod_batch_calc.query('run_date_ym == 202508').copy()

df["oop_target_bad"] = 1 - df["oop_target"]

df["osbal_current_bad"] = df["oop_target_bad"] * df["osbal_as_of_current_date"]

pt = pd.pivot_table(
    df,
    index="oop_risk_segment_prod",
    values=["oop_target_bad", "osbal_as_of_resignation_date", "osbal_current_bad"],
    aggfunc={
        "oop_target_bad": ["count", "mean"],
        "osbal_as_of_resignation_date": "sum",
        "osbal_current_bad": "sum"
    }
)

pt[("php_weighted_outstanding_bad_rate", "ratio")] = (
    pt[("osbal_current_bad", "sum")] / pt[("osbal_as_of_resignation_date", "sum")]
)

pt.sort_index(ascending=True)

oop_target_bad           osbal_as_of_resignation_date  \
                               count      mean                          sum   
oop_risk_segment_prod                                                         
A                                 82  0.536585                   996722.658   
B                                573  0.596859                  8030013.185   
C                                726  0.629477                  9880118.211   
D                                602  0.765781                  8332637.023   
E                                229  0.781659                  2511201.696   
F                                200      0.83                  1915113.287   

                      osbal_current_bad php_weighted_outstanding_bad_rate  
                                    sum                             ratio  
oop_risk_segment_prod                                                      
A                            788991.118                          0.791585  
B                           6077774.472                          0.756882  
C                           7649651.196                          0.774247  
D                             7319197.1                          0.878377  
E                           2352701.753                          0.936883  
F                           1864303.486                          0.973469

In [95]:
# Existing prod users, BS score, June
df = dt_prod_batch_calc.query('run_date_ym == 202506').copy()

df["oop_target_bad"] = 1 - df["oop_target"]

df["osbal_current_bad"] = df["oop_target_bad"] * df["osbal_as_of_current_date"]

pt = pd.pivot_table(
    df,
    index="oop_risk_segment_bs",
    values=["oop_target_bad", "osbal_as_of_resignation_date", "osbal_current_bad"],
    aggfunc={
        "oop_target_bad": ["count", "mean"],
        "osbal_as_of_resignation_date": "sum",
        "osbal_current_bad": "sum"
    }
)

pt[("php_weighted_outstanding_bad_rate", "ratio")] = (
    pt[("osbal_current_bad", "sum")] / pt[("osbal_as_of_resignation_date", "sum")]
)

pt.sort_index(ascending=False)

oop_target_bad           osbal_as_of_resignation_date  \
                             count      mean                          sum   
oop_risk_segment_bs                                                         
A                               55       0.6                  1345243.152   
B                              314  0.576433                  4597437.804   
C                              383  0.751958                  5800812.650   
D                              345   0.77971                  4279847.142   
E                              190  0.842105                  1885885.258   
F                              133  0.834586                  1424158.834   

                    osbal_current_bad php_weighted_outstanding_bad_rate  
                                  sum                             ratio  
oop_risk_segment_bs                                                      
A                          576780.653                          0.428756  
B                         3432070.838                          0.746518  
C                         4582259.904                          0.789934  
D                         3650398.723                          0.852927  
E                         1723122.854                          0.913694  
F                         1304717.646                          0.916132

In [96]:
# Existing prod users, BS score, July
df = dt_prod_batch_calc.query('run_date_ym == 202507').copy()

df["oop_target_bad"] = 1 - df["oop_target"]

df["osbal_current_bad"] = df["oop_target_bad"] * df["osbal_as_of_current_date"]

pt = pd.pivot_table(
    df,
    index="oop_risk_segment_bs",
    values=["oop_target_bad", "osbal_as_of_resignation_date", "osbal_current_bad"],
    aggfunc={
        "oop_target_bad": ["count", "mean"],
        "osbal_as_of_resignation_date": "sum",
        "osbal_current_bad": "sum"
    }
)

pt[("php_weighted_outstanding_bad_rate", "ratio")] = (
    pt[("osbal_current_bad", "sum")] / pt[("osbal_as_of_resignation_date", "sum")]
)

pt.sort_index(ascending=False)

oop_target_bad           osbal_as_of_resignation_date  \
                             count      mean                          sum   
oop_risk_segment_bs                                                         
A                               62  0.516129                  1103816.065   
B                              493  0.557809                  7427358.165   
C                              601  0.678869                  9773053.029   
D                              495  0.741414                  6459836.382   
E                              266   0.81203                  3048898.348   
F                              224  0.839286                  2228561.216   

                    osbal_current_bad php_weighted_outstanding_bad_rate  
                                  sum                             ratio  
oop_risk_segment_bs                                                      
A                          678824.664                           0.61498  
B                          5301021.01                          0.713716  
C                         8281634.179                          0.847395  
D                         5510795.594                          0.853086  
E                         2902747.121                          0.952064  
F                         2320187.422                          1.041115

In [97]:
# Existing prod users, BS score, Aug
df = dt_prod_batch_calc.query('run_date_ym == 202508').copy()

df["oop_target_bad"] = 1 - df["oop_target"]

df["osbal_current_bad"] = df["oop_target_bad"] * df["osbal_as_of_current_date"]

pt = pd.pivot_table(
    df,
    index="oop_risk_segment_bs",
    values=["oop_target_bad", "osbal_as_of_resignation_date", "osbal_current_bad"],
    aggfunc={
        "oop_target_bad": ["count", "mean"],
        "osbal_as_of_resignation_date": "sum",
        "osbal_current_bad": "sum"
    }
)

pt[("php_weighted_outstanding_bad_rate", "ratio")] = (
    pt[("osbal_current_bad", "sum")] / pt[("osbal_as_of_resignation_date", "sum")]
)

pt.sort_index(ascending=False)

oop_target_bad           osbal_as_of_resignation_date  \
                             count      mean                          sum   
oop_risk_segment_bs                                                         
A                               59  0.423729                   769015.938   
B                              436  0.559633                  5867685.903   
C                              509  0.642436                  8137019.938   
D                              443  0.722348                  5536531.567   
E                              229  0.781659                  2405243.839   
F                              191  0.863874                  1816029.988   

                    osbal_current_bad php_weighted_outstanding_bad_rate  
                                  sum                             ratio  
oop_risk_segment_bs                                                      
A                          523424.259                          0.680642  
B                         4492164.357                          0.765577  
C                         6644261.503                          0.816547  
D                         4754393.515                          0.858731  
E                         2397315.966                          0.996704  
F                         1980813.816                          1.090738

### bs existing

In [98]:
dt_bs_oop_ex_calc["oop_risk_segment_bs"] = pd.cut(
    dt_bs_oop_ex_calc["score_oop"],
    bins=edges_ex,
    labels=labels_ex,
    right=False,          # intervals are [a, b) so boundary goes to the higher segment
    include_lowest=True,
)

C:\Users\Dwaipayan\AppData\Local\Temp\ipykernel_61508\125666790.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dt_bs_oop_ex_calc["oop_risk_segment_bs"] = pd.cut(


In [99]:
# Existing BS users, BS score, June
df = dt_bs_oop_ex_calc.query('calc_date_ym == 202506').copy()

df["oop_target_bad"] = 1 - df["oop_target"]

df["osbal_current_bad"] = df["oop_target_bad"] * df["osbal_as_of_current_date"]

pt = pd.pivot_table(
    df,
    index="oop_risk_segment_bs",
    values=["oop_target_bad", "osbal_as_of_resignation_date", "osbal_current_bad"],
    aggfunc={
        "oop_target_bad": ["count", "mean"],
        "osbal_as_of_resignation_date": "sum",
        "osbal_current_bad": "sum"
    }
)

pt[("php_weighted_outstanding_bad_rate", "ratio")] = (
    pt[("osbal_current_bad", "sum")] / pt[("osbal_as_of_resignation_date", "sum")]
)

pt.sort_index(ascending=False)

oop_target_bad           osbal_as_of_resignation_date  \
                             count      mean                          sum   
oop_risk_segment_bs                                                         
A                              138  0.543478                 2.209076e+06   
B                              977   0.65609                 1.326482e+07   
C                             1240  0.776613                 1.749603e+07   
D                              981  0.796126                 1.170957e+07   
E                              527   0.86148                 5.724098e+06   
F                              407  0.874693                 4.310766e+06   

                    osbal_current_bad php_weighted_outstanding_bad_rate  
                                  sum                             ratio  
oop_risk_segment_bs                                                      
A                         1072506.209                            0.4855  
B                         9805358.873                            0.7392  
C                        14509473.333                          0.829301  
D                         9657150.403                          0.824723  
E                           5295783.1                          0.925173  
F                         4065479.905                          0.943099

In [100]:
# Existing BS users, BS score, July
df = dt_bs_oop_ex_calc.query('calc_date_ym == 202507').copy()

df["oop_target_bad"] = 1 - df["oop_target"]

df["osbal_current_bad"] = df["oop_target_bad"] * df["osbal_as_of_current_date"]

pt = pd.pivot_table(
    df,
    index="oop_risk_segment_bs",
    values=["oop_target_bad", "osbal_as_of_resignation_date", "osbal_current_bad"],
    aggfunc={
        "oop_target_bad": ["count", "mean"],
        "osbal_as_of_resignation_date": "sum",
        "osbal_current_bad": "sum"
    }
)

pt[("php_weighted_outstanding_bad_rate", "ratio")] = (
    pt[("osbal_current_bad", "sum")] / pt[("osbal_as_of_resignation_date", "sum")]
)

pt.sort_index(ascending=False)

oop_target_bad           osbal_as_of_resignation_date  \
                             count      mean                          sum   
oop_risk_segment_bs                                                         
A                              185  0.459459                 2.591259e+06   
B                             1354  0.616691                 1.729851e+07   
C                             1483  0.719488                 1.957808e+07   
D                              867  0.778547                 1.026759e+07   
E                              414   0.84058                 4.137840e+06   
F                              335  0.862687                 3.389232e+06   

                    osbal_current_bad php_weighted_outstanding_bad_rate  
                                  sum                             ratio  
oop_risk_segment_bs                                                      
A                         1236015.341                          0.476994  
B                        11967777.448                          0.691839  
C                        15443157.618                          0.788798  
D                         8725294.208                           0.84979  
E                         4057704.405                          0.980633  
F                         3309527.956                          0.976483

In [101]:
# Existing BS users, BS score, Aug
df = dt_bs_oop_ex_calc.query('calc_date_ym == 202508').copy()

df["oop_target_bad"] = 1 - df["oop_target"]

df["osbal_current_bad"] = df["oop_target_bad"] * df["osbal_as_of_current_date"]

pt = pd.pivot_table(
    df,
    index="oop_risk_segment_bs",
    values=["oop_target_bad", "osbal_as_of_resignation_date", "osbal_current_bad"],
    aggfunc={
        "oop_target_bad": ["count", "mean"],
        "osbal_as_of_resignation_date": "sum",
        "osbal_current_bad": "sum"
    }
)

pt[("php_weighted_outstanding_bad_rate", "ratio")] = (
    pt[("osbal_current_bad", "sum")] / pt[("osbal_as_of_resignation_date", "sum")]
)

pt.sort_index(ascending=False)

oop_target_bad           osbal_as_of_resignation_date  \
                             count      mean                          sum   
oop_risk_segment_bs                                                         
A                              177  0.435028                 2.137342e+06   
B                             1227  0.593317                 1.411636e+07   
C                             1354  0.693501                 1.738099e+07   
D                              790  0.758228                 8.958886e+06   
E                              355  0.822535                 3.559436e+06   
F                              312  0.884615                 2.903849e+06   

                    osbal_current_bad php_weighted_outstanding_bad_rate  
                                  sum                             ratio  
oop_risk_segment_bs                                                      
A                         1026499.742                          0.480269  
B                        10161212.979                          0.719818  
C                        14009786.738                          0.806041  
D                         7977163.997                          0.890419  
E                         3678066.458                          1.033328  
F                         3211474.534                          1.105937

# Attrition

In [102]:
# BS Attrition
sql_query_attrition = """
SELECT *
FROM `prj-prod-dataplatform.risk_mart.tendo_backscored_jan24_jan26_20260201_attrition`
"""

dt_bs_attr = client.query(sql_query_attrition).to_dataframe()

In [103]:
dt_bs_attr['ee_customer_id'] = dt_bs_attr['ee_customer_id'].astype('str')

dt_bs_attr["calc_date"] = pd.to_datetime(dt_bs_attr["calc_date"], errors="coerce")
dt_bs_attr["calc_date_correct"] = pd.to_datetime(dt_bs_attr["calc_date"], errors="coerce") - pd.DateOffset(months=1)
dt_bs_attr['calc_date_ym'] = dt_bs_attr['calc_date_correct'].dt.year*100 + dt_bs_attr['calc_date_correct'].dt.month

In [104]:
new_1m = dt_bs_attr['ee_onboarding_month'] == dt_bs_attr['calc_date_ym']
dt_bs_attr['is_new_customer_flag_1m'] = new_1m.astype('int')

In [105]:
dt_bs_attr.head(2)

,ee_customer_id,calc_date,ee_onboarding_month,is_new_customer_flag_3m,target_event,target_duration_dev,score_attr,score_attr_segment,calc_date_correct,calc_date_ym,is_new_customer_flag_1m
0,1021876,2024-01-01,202309,0,1.0,3.0,2.0,Very high,2023-12-01,202312,0
1,1041967,2024-01-01,202307,0,0.0,12.0,2.0,Very high,2023-12-01,202312,0


In [107]:
dt_res = pd.read_pickle(
    generate_bucket_url('Oleh/tendo/data/resignation_data_14012026.pkl', GS_BUCKET)
)

In [108]:
dt_res.head(1)

,ee_customer_id,ee_resignation_date_correct
0,1165907,NaT


## Distribution metrics

### prod new

In [109]:
dt_prod_api.shape

(161175, 15)

In [110]:
dt_prod_api.head()

,ee_customer_id,run_date,attrition_risk_segment,attrition_time_to_leave,oop_score_prod,oop_risk_segment_prod,ee_onboarding_date,oop_target,score_oop,osbal_as_of_resignation_date,osbal_as_of_oop_eligible_date,osbal_as_of_current_date,onb_rd_diff,onboarding_date_ym,run_date_ym
0,1532098,2026-01-13,10-12,10,0.368567,E,NaT,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,202601
1,1532098,2026-01-13,10-12,10,0.368567,E,NaT,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,202601
2,1532098,2026-01-13,10-12,10,0.368567,E,NaT,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,202601
3,1648044,2026-01-13,10-12,10,0.382697,E,NaT,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,202601
4,1648044,2026-01-13,10-12,10,0.382697,E,NaT,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,202601


In [111]:
dt_prod_api_attr = dt_prod_api.merge(
    dt_res,
    how='left',
    on='ee_customer_id'
).merge(
    dt_bs_attr[['ee_customer_id', 'calc_date_ym', 'score_attr', 'score_attr_segment', 'is_new_customer_flag_1m']],
    how='left',
    left_on=['ee_customer_id','run_date_ym'],
    right_on=['ee_customer_id','calc_date_ym']
)

In [112]:
dt_prod_api_calc = (dt_prod_api_attr
                    .dropna(subset=['ee_onboarding_date'])
                    .sort_values(['onb_rd_diff','run_date'])
                    .drop_duplicates(subset=['ee_customer_id'], keep='first'))

In [113]:
months_diff = (
    (dt_prod_api_calc['ee_resignation_date_correct'].dt.year - dt_prod_api_calc['run_date'].dt.year) * 12 +
    (dt_prod_api_calc['ee_resignation_date_correct'].dt.month - dt_prod_api_calc['run_date'].dt.month)
)

dt_prod_api_calc['time_to_attrition'] = np.where(dt_prod_api_calc['ee_resignation_date_correct'].isna(), np.nan, months_diff)
dt_prod_api_calc['attrition_event'] = dt_prod_api_calc['ee_resignation_date_correct'].notna().astype('int')

In [114]:
dt_prod_api_calc['score_attr_corrected'] = dt_prod_api_calc['score_attr'].replace(np.inf, 15)
dt_prod_api_calc['attrition_score_prod'] = dt_prod_api_calc['attrition_time_to_leave'].replace('12+', '15').astype('float')

mapping_dict = {
            1: 'Very high',
            2: 'Very high',
            3: 'Very high',
            4: 'High',
            5: 'High',
            6: 'High',
            7: 'Average',
            8: 'Average',
            9: 'Average',
            10: 'Low',
            11: 'Low',
            12: 'Low',
            15: 'Very low'
        }

dt_prod_api_calc['attrition_risk_segment_prod'] = dt_prod_api_calc['attrition_score_prod'].replace(mapping_dict)

In [115]:
# New prod users, prod score, June-Aug
df = dt_prod_api_calc.query('onboarding_date_ym >= 202506 & onboarding_date_ym <= 202508').copy()

pt = pd.pivot_table(
    df,
    index="attrition_risk_segment_prod", # score_attr_segment - BS
    values=["attrition_event", "attrition_score_prod", "time_to_attrition", "ee_customer_id"],
    aggfunc={
        "attrition_event": "mean",
        "attrition_score_prod": "mean",
        "time_to_attrition": "mean",
        "ee_customer_id": "count"
    }
).rename(
    columns={"ee_customer_id": "count"}
)[['attrition_event', 'attrition_score_prod', 'time_to_attrition', 'count']]

order = ['Very low', 'Low', 'Average', 'High', 'Very high']

pt.loc[order]

,attrition_event,attrition_score_prod,time_to_attrition,count
attrition_risk_segment_prod,,,,
Very low,0.162791,15.000000,0.571429,43
Low,0.135135,10.254054,3.320000,185
Average,0.120253,8.645570,2.894737,158
High,0.134259,4.450000,2.268966,1080
Very high,0.152038,2.929467,1.896907,638


In [116]:
# New prod users, BS score, June-Aug
df = dt_prod_api_calc.query('onboarding_date_ym >= 202506 & onboarding_date_ym <= 202508').copy()

pt = pd.pivot_table(
    df,
    index="score_attr_segment", # score_attr_segment - BS
    values=["attrition_event", "score_attr_corrected", "time_to_attrition", "ee_customer_id"],
    aggfunc={
        "attrition_event": "mean",
        "score_attr_corrected": "mean",
        "time_to_attrition": "mean",
        "ee_customer_id": "count"
    }
).rename(
    columns={"ee_customer_id": "count"}
)[['attrition_event', 'score_attr_corrected', 'time_to_attrition', 'count']]

order = ['Very low', 'Low', 'Average', 'High', 'Very high']

pt.loc[order]

,attrition_event,score_attr_corrected,time_to_attrition,count
score_attr_segment,,,,
Very low,0.118182,15.000000,3.666667,330
Low,0.146341,10.329268,2.666667,164
Average,0.103774,7.896226,2.712121,636
High,0.139810,5.220379,2.737288,844
Very high,0.152542,2.949153,2.555556,59


### bs new

In [117]:
dt_bs_attr.head()

,ee_customer_id,calc_date,ee_onboarding_month,is_new_customer_flag_3m,target_event,target_duration_dev,score_attr,score_attr_segment,calc_date_correct,calc_date_ym,is_new_customer_flag_1m
0,1021876,2024-01-01,202309,0,1.0,3.0,2.0,Very high,2023-12-01,202312,0
1,1041967,2024-01-01,202307,0,0.0,12.0,2.0,Very high,2023-12-01,202312,0
2,1070864,2024-01-01,202312,1,0.0,12.0,2.0,Very high,2023-12-01,202312,1
3,1080964,2024-01-01,202307,0,0.0,12.0,2.0,Very high,2023-12-01,202312,0
4,1081140,2024-01-01,202307,0,1.0,7.0,2.0,Very high,2023-12-01,202312,0


In [118]:
dt_bs_attr_dev = dt_bs_attr.merge(
    dt_res,
    how='left',
    on='ee_customer_id'
)

In [119]:
dt_bs_attr_dev_calc = dt_bs_attr_dev.query('is_new_customer_flag_1m == 1').copy()

In [120]:
months_diff = (
    (dt_bs_attr_dev_calc['ee_resignation_date_correct'].dt.year - dt_bs_attr_dev_calc['calc_date_correct'].dt.year) * 12 +
    (dt_bs_attr_dev_calc['ee_resignation_date_correct'].dt.month - dt_bs_attr_dev_calc['calc_date_correct'].dt.month)
)

dt_bs_attr_dev_calc['time_to_attrition'] = np.where(dt_bs_attr_dev_calc['ee_resignation_date_correct'].isna(), np.nan, months_diff)
dt_bs_attr_dev_calc['attrition_event'] = dt_bs_attr_dev_calc['ee_resignation_date_correct'].notna().astype('int')

In [121]:
dt_bs_attr_dev_calc['score_attr_corrected'] = dt_bs_attr_dev_calc['score_attr'].replace(np.inf, 15)

In [122]:
# New BS users, BS score, June-Aug
df = dt_bs_attr_dev_calc.query('ee_onboarding_month >= 202506 & ee_onboarding_month <= 202508').copy()

pt = pd.pivot_table(
    df,
    index="score_attr_segment",
    values=["attrition_event", "score_attr_corrected", "time_to_attrition", "ee_customer_id"],
    aggfunc={
        "attrition_event": "mean",
        "score_attr_corrected": "mean",
        "time_to_attrition": "mean",
        "ee_customer_id": "count"
    }
).rename(
    columns={"ee_customer_id": "count"}
)[['attrition_event', 'score_attr_corrected', 'time_to_attrition', 'count']]

order = ['Very low', 'Low', 'Average', 'High', 'Very high']

pt.loc[order]

C:\Users\Dwaipayan\AppData\Local\Temp\ipykernel_61508\1994193056.py:2: RuntimeWarning: Engine has switched to 'python' because numexpr does not support extension array dtypes. Please set your engine to python manually.
  df = dt_bs_attr_dev_calc.query('ee_onboarding_month >= 202506 & ee_onboarding_month <= 202508').copy()


,attrition_event,score_attr_corrected,time_to_attrition,count
score_attr_segment,,,,
Very low,0.079256,15.000000,3.759259,2044
Low,0.079388,10.475601,3.064220,2746
Average,0.129822,8.133050,3.051047,5885
High,0.171187,5.507221,2.808642,2839
Very high,0.255814,2.574419,2.809091,430


### prod existing

In [123]:
dt_prod_batch.shape

(384214, 16)

In [124]:
dt_prod_batch.head()

,ee_customer_id,run_date,attrition_risk_segment,attrition_time_to_leave,oop_score_prod,oop_risk_segment_prod,run_date_ym,ee_onboarding_date,oop_target,osbal_as_of_resignation_date,osbal_as_of_oop_eligible_date,osbal_as_of_current_date,calc_date_ym,score_oop,onboarding_date_ym,onb_rd_diff
0,1157338,2025-10-14,Low,10,0.704469,A,202510,2024-02-14,<NA>,NaN,NaN,130729.530,202510.0,0.691760,202402.0,608.0
1,1456942,2025-10-14,Low,10,0.735589,A,202510,2025-06-14,<NA>,NaN,NaN,62474.122,202510.0,0.735589,202506.0,122.0
2,1318047,2025-10-14,Low,11,0.758049,A,202510,2024-12-14,<NA>,NaN,NaN,0.000,202510.0,0.758049,202412.0,304.0
3,1167047,2025-10-14,Low,12,0.726716,A,202510,2024-03-14,<NA>,NaN,NaN,0.000,202510.0,0.726716,202403.0,579.0
4,1343065,2025-10-14,Low,12,0.758949,A,202510,2025-02-14,<NA>,NaN,NaN,0.000,202510.0,0.758949,202502.0,242.0


In [125]:
dt_prod_batch_attr = dt_prod_batch.merge(
    dt_res,
    how='left',
    on='ee_customer_id'
).merge(
    dt_bs_attr[['ee_customer_id', 'calc_date_ym', 'score_attr', 'score_attr_segment', 'is_new_customer_flag_1m']],
    how='left',
    left_on=['ee_customer_id','run_date_ym'],
    right_on=['ee_customer_id','calc_date_ym']
)

In [126]:
dt_prod_batch_attr.shape

(384214, 21)

In [127]:
dt_prod_batch_calc = dt_prod_batch_attr.dropna(subset=['ee_onboarding_date'])

In [129]:
months_diff = (
    (dt_prod_batch_calc['ee_resignation_date_correct'].dt.year - dt_prod_batch_calc['run_date'].dt.year) * 12 +
    (dt_prod_batch_calc['ee_resignation_date_correct'].dt.month - dt_prod_batch_calc['run_date'].dt.month)
)

dt_prod_batch_calc['time_to_attrition'] = np.where(dt_prod_batch_calc['ee_resignation_date_correct'].isna(), np.nan, months_diff)
dt_prod_batch_calc['attrition_event'] = dt_prod_batch_calc['ee_resignation_date_correct'].notna().astype('int')

C:\Users\Dwaipayan\AppData\Local\Temp\ipykernel_61508\688793827.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dt_prod_batch_calc['time_to_attrition'] = np.where(dt_prod_batch_calc['ee_resignation_date_correct'].isna(), np.nan, months_diff)
C:\Users\Dwaipayan\AppData\Local\Temp\ipykernel_61508\688793827.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dt_prod_batch_calc['attrition_event'] = dt_prod_batch_calc['ee_resignation_date_correct'].notna().astype('int')


In [130]:
dt_prod_batch_calc['score_attr_corrected'] = dt_prod_batch_calc['score_attr'].replace(np.inf, 15)
dt_prod_batch_calc['attrition_score_prod'] = dt_prod_batch_calc['attrition_time_to_leave'].replace('12+', '15').astype('float')

mapping_dict = {
            1: 'Very high',
            2: 'Very high',
            3: 'Very high',
            4: 'High',
            5: 'High',
            6: 'High',
            7: 'Average',
            8: 'Average',
            9: 'Average',
            10: 'Low',
            11: 'Low',
            12: 'Low',
            15: 'Very low'
        }

dt_prod_batch_calc['attrition_risk_segment_prod'] = dt_prod_batch_calc['attrition_score_prod'].replace(mapping_dict)

C:\Users\Dwaipayan\AppData\Local\Temp\ipykernel_61508\2571472067.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dt_prod_batch_calc['score_attr_corrected'] = dt_prod_batch_calc['score_attr'].replace(np.inf, 15)
C:\Users\Dwaipayan\AppData\Local\Temp\ipykernel_61508\2571472067.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dt_prod_batch_calc['attrition_score_prod'] = dt_prod_batch_calc['attrition_time_to_leave'].replace('12+', '15').astype('float')
C:\Users\Dwaipayan\AppData\Local\Temp\ipykernel_6150

In [131]:
# Existing prod users, prod score, June
df = dt_prod_batch_calc.query('run_date_ym == 202506').copy()

pt = pd.pivot_table(
    df,
    index="attrition_risk_segment_prod", # score_attr_segment - BS
    values=["attrition_event", "attrition_score_prod", "time_to_attrition", "ee_customer_id"],
    aggfunc={
        "attrition_event": "mean",
        "attrition_score_prod": "mean",
        "time_to_attrition": "mean",
        "ee_customer_id": "count"
    }
).rename(
    columns={"ee_customer_id": "count"}
)[['attrition_event', 'attrition_score_prod', 'time_to_attrition', 'count']]

order = ['Very low', 'Low', 'Average', 'High', 'Very high']

pt.loc[order]

,attrition_event,attrition_score_prod,time_to_attrition,count
attrition_risk_segment_prod,,,,
Very low,0.093900,15.000000,3.136752,4984
Low,0.087040,10.613582,2.752747,2091
Average,0.134625,8.030628,1.942159,5779
High,0.191105,5.356121,2.574013,6363
Very high,0.324153,2.891949,2.392157,472


In [132]:
# Existing prod users, prod score, July
df = dt_prod_batch_calc.query('run_date_ym == 202507').copy()

pt = pd.pivot_table(
    df,
    index="attrition_risk_segment_prod", # score_attr_segment - BS
    values=["attrition_event", "attrition_score_prod", "time_to_attrition", "ee_customer_id"],
    aggfunc={
        "attrition_event": "mean",
        "attrition_score_prod": "mean",
        "time_to_attrition": "mean",
        "ee_customer_id": "count"
    }
).rename(
    columns={"ee_customer_id": "count"}
)[['attrition_event', 'attrition_score_prod', 'time_to_attrition', 'count']]

order = ['Very low', 'Low', 'Average', 'High', 'Very high']

pt.loc[order]

,attrition_event,attrition_score_prod,time_to_attrition,count
attrition_risk_segment_prod,,,,
Very low,0.082086,15.000000,2.551020,8954
Low,0.079343,10.443213,1.867830,5054
Average,0.113185,8.105433,1.742021,13288
High,0.169955,5.500787,2.104701,8261
Very high,0.291339,3.000000,2.324324,127


In [133]:
# Existing prod users, prod score, August
df = dt_prod_batch_calc.query('run_date_ym == 202508').copy()

pt = pd.pivot_table(
    df,
    index="attrition_risk_segment_prod", # score_attr_segment - BS
    values=["attrition_event", "attrition_score_prod", "time_to_attrition", "ee_customer_id"],
    aggfunc={
        "attrition_event": "mean",
        "attrition_score_prod": "mean",
        "time_to_attrition": "mean",
        "ee_customer_id": "count"
    }
).rename(
    columns={"ee_customer_id": "count"}
)[['attrition_event', 'attrition_score_prod', 'time_to_attrition', 'count']]

order = ['Very low', 'Low', 'Average', 'High', 'Very high']

pt.loc[order]

,attrition_event,attrition_score_prod,time_to_attrition,count
attrition_risk_segment_prod,,,,
Very low,0.071177,15.000000,2.061329,8247
Low,0.064304,10.428627,1.189759,5163
Average,0.086146,8.115601,1.359966,13512
High,0.131978,5.511141,1.730056,8168
Very high,0.230769,3.000000,2.095238,91


In [134]:
# Existing prod users, BS score, June
df = dt_prod_batch_calc.query('run_date_ym == 202506').copy()

pt = pd.pivot_table(
    df,
    index="score_attr_segment", # score_attr_segment - BS
    values=["attrition_event", "score_attr_corrected", "time_to_attrition", "ee_customer_id"],
    aggfunc={
        "attrition_event": "mean",
        "score_attr_corrected": "mean",
        "time_to_attrition": "mean",
        "ee_customer_id": "count"
    }
).rename(
    columns={"ee_customer_id": "count"}
)[['attrition_event', 'score_attr_corrected', 'time_to_attrition', 'count']]

order = ['Very low', 'Low', 'Average', 'High', 'Very high']

pt.loc[order]

,attrition_event,score_attr_corrected,time_to_attrition,count
score_attr_segment,,,,
Very low,0.085591,15.000000,3.425577,5573
Low,0.099319,10.443332,3.419355,2497
Average,0.128749,8.116943,3.111524,6268
High,0.190585,5.393251,2.890710,4801
Very high,0.285714,2.907143,3.087500,280


In [135]:
# Existing prod users, BS score, July
df = dt_prod_batch_calc.query('run_date_ym == 202507').copy()

pt = pd.pivot_table(
    df,
    index="score_attr_segment", # score_attr_segment - BS
    values=["attrition_event", "score_attr_corrected", "time_to_attrition", "ee_customer_id"],
    aggfunc={
        "attrition_event": "mean",
        "score_attr_corrected": "mean",
        "time_to_attrition": "mean",
        "ee_customer_id": "count"
    }
).rename(
    columns={"ee_customer_id": "count"}
)[['attrition_event', 'score_attr_corrected', 'time_to_attrition', 'count']]

order = ['Very low', 'Low', 'Average', 'High', 'Very high']

pt.loc[order]

,attrition_event,score_attr_corrected,time_to_attrition,count
score_attr_segment,,,,
Very low,0.073187,15.000000,2.911411,9100
Low,0.073016,10.406746,2.782609,5040
Average,0.094770,8.129714,2.792545,13021
High,0.142007,5.516425,2.714026,7732
Very high,0.262136,3.000000,3.074074,103


In [136]:
# Existing prod users, BS score, August
df = dt_prod_batch_calc.query('run_date_ym == 202508').copy()

pt = pd.pivot_table(
    df,
    index="score_attr_segment", # score_attr_segment - BS
    values=["attrition_event", "score_attr_corrected", "time_to_attrition", "ee_customer_id"],
    aggfunc={
        "attrition_event": "mean",
        "score_attr_corrected": "mean",
        "time_to_attrition": "mean",
        "ee_customer_id": "count"
    }
).rename(
    columns={"ee_customer_id": "count"}
)[['attrition_event', 'score_attr_corrected', 'time_to_attrition', 'count']]

order = ['Very low', 'Low', 'Average', 'High', 'Very high']

pt.loc[order]

,attrition_event,score_attr_corrected,time_to_attrition,count
score_attr_segment,,,,
Very low,0.061388,15.000000,2.484436,8373
Low,0.057876,10.400557,2.319588,5028
Average,0.073575,8.132015,2.310591,13347
High,0.112680,5.525118,2.230248,7863
Very high,0.227848,3.000000,2.444444,79


### bs existing

In [137]:
dt_bs_attr_dev_calc.head()

,ee_customer_id,calc_date,ee_onboarding_month,is_new_customer_flag_3m,target_event,target_duration_dev,score_attr,score_attr_segment,calc_date_correct,calc_date_ym,is_new_customer_flag_1m,ee_resignation_date_correct,time_to_attrition,attrition_event,score_attr_corrected
2,1070864,2024-01-01,202312,1,0.0,12.0,2.0,Very high,2023-12-01,202312,1,2025-07-11,19.0,1,2.0
39,1126217,2024-01-01,202312,1,1.0,1.0,2.0,Very high,2023-12-01,202312,1,2024-01-31,1.0,1,2.0
41,1126627,2024-01-01,202312,1,1.0,1.0,2.0,Very high,2023-12-01,202312,1,2024-02-11,2.0,1,2.0
42,1126761,2024-01-01,202312,1,0.0,12.0,2.0,Very high,2023-12-01,202312,1,NaT,NaN,0,2.0
44,1128280,2024-01-01,202312,1,0.0,12.0,2.0,Very high,2023-12-01,202312,1,NaT,NaN,0,2.0


In [138]:
dt_bs_attr_dev_calc.shape

(145009, 15)

In [139]:
dt_bs_attr_dev_calc = dt_bs_attr_dev.query('is_new_customer_flag_3m == 0').copy()

C:\Users\Dwaipayan\AppData\Local\Temp\ipykernel_61508\3206105625.py:1: RuntimeWarning: Engine has switched to 'python' because numexpr does not support extension array dtypes. Please set your engine to python manually.
  dt_bs_attr_dev_calc = dt_bs_attr_dev.query('is_new_customer_flag_3m == 0').copy()


In [140]:
months_diff = (
    (dt_bs_attr_dev_calc['ee_resignation_date_correct'].dt.year - dt_bs_attr_dev_calc['calc_date_correct'].dt.year) * 12 +
    (dt_bs_attr_dev_calc['ee_resignation_date_correct'].dt.month - dt_bs_attr_dev_calc['calc_date_correct'].dt.month)
)

dt_bs_attr_dev_calc['time_to_attrition'] = np.where(dt_bs_attr_dev_calc['ee_resignation_date_correct'].isna(), np.nan, months_diff)
dt_bs_attr_dev_calc['attrition_event'] = dt_bs_attr_dev_calc['ee_resignation_date_correct'].notna().astype('int')

In [141]:
dt_bs_attr_dev_calc['score_attr_corrected'] = dt_bs_attr_dev_calc['score_attr'].replace(np.inf, 15)

In [142]:
# Existing BS users, BS score, June
df = dt_bs_attr_dev_calc.query('calc_date_ym == 202506').copy()

pt = pd.pivot_table(
    df,
    index="score_attr_segment",
    values=["attrition_event", "score_attr_corrected", "time_to_attrition", "ee_customer_id"],
    aggfunc={
        "attrition_event": "mean",
        "score_attr_corrected": "mean",
        "time_to_attrition": "mean",
        "ee_customer_id": "count"
    }
).rename(
    columns={"ee_customer_id": "count"}
)[['attrition_event', 'score_attr_corrected', 'time_to_attrition', 'count']]

order = ['Very low', 'Low', 'Average', 'High', 'Very high']

pt.loc[order]

,attrition_event,score_attr_corrected,time_to_attrition,count
score_attr_segment,,,,
Very low,0.095607,15.000000,3.305405,19350
Low,0.102262,10.512502,3.024580,7559
Average,0.130475,8.130360,2.963484,17421
High,0.183858,5.431248,3.009388,12167
Very high,0.296367,2.925430,3.019355,523


In [143]:
# Existing BS users, BS score, July
df = dt_bs_attr_dev_calc.query('calc_date_ym == 202507').copy()

pt = pd.pivot_table(
    df,
    index="score_attr_segment",
    values=["attrition_event", "score_attr_corrected", "time_to_attrition", "ee_customer_id"],
    aggfunc={
        "attrition_event": "mean",
        "score_attr_corrected": "mean",
        "time_to_attrition": "mean",
        "ee_customer_id": "count"
    }
).rename(
    columns={"ee_customer_id": "count"}
)[['attrition_event', 'score_attr_corrected', 'time_to_attrition', 'count']]

order = ['Very low', 'Low', 'Average', 'High', 'Very high']

pt.loc[order]

,attrition_event,score_attr_corrected,time_to_attrition,count
score_attr_segment,,,,
Very low,0.076149,15.000000,2.996867,20959
Low,0.071939,10.478317,2.967904,13859
Average,0.107573,8.167364,2.813938,27479
High,0.150465,5.510463,2.726512,14289
Very high,0.253796,2.932755,2.461538,461


In [144]:
# Existing BS users, BS score, August
df = dt_bs_attr_dev_calc.query('calc_date_ym == 202508').copy()

pt = pd.pivot_table(
    df,
    index="score_attr_segment",
    values=["attrition_event", "score_attr_corrected", "time_to_attrition", "ee_customer_id"],
    aggfunc={
        "attrition_event": "mean",
        "score_attr_corrected": "mean",
        "time_to_attrition": "mean",
        "ee_customer_id": "count"
    }
).rename(
    columns={"ee_customer_id": "count"}
)[['attrition_event', 'score_attr_corrected', 'time_to_attrition', 'count']]

order = ['Very low', 'Low', 'Average', 'High', 'Very high']

pt.loc[order]

,attrition_event,score_attr_corrected,time_to_attrition,count
score_attr_segment,,,,
Very low,0.060132,15.000000,2.589564,21353
Low,0.058641,10.480846,2.513203,14853
Average,0.088052,8.167436,2.354618,29880
High,0.122879,5.519820,2.270197,14909
Very high,0.232184,2.875862,1.821782,435
